# Занятие 11. Методы машинного обучения

In [ ]:
# 📥 Установка пакетов и библиотек
!pip install pandas
!pip install matplotlib
!pip install scikit-learn
!pip install plotly
!pip install rdkit 
!pip install torch
!pip install networkx

In [ ]:
# 🛠️ Загрузка необходимых пакетов в среду выполнения
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import copy
import networkx as nx

import sklearn 

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import Draw, AllChem, Descriptors, rdFingerprintGenerator, rdMolDescriptors, rdMolTransforms, Fragments, rdFreeSASA, rdMIF, Descriptors3D
from rdkit.Chem import MACCSkeys

# show()
REPO = "/home/jupyter/project/computational-chemistry-course"
sys.path.append(os.path.join(REPO, "tools"))
import ml_tool
import md_tools

# 🧮 Глава 1. Молекулярные дескрипторы

Ранее мы уже использовали матрицу признаков $X$ размера $n \times m$, в которой строки являлись объектами, а столбцы - признаками. Признаки молекул можно описать **молекулярными дескрипторами**. Дескрипторы различаются тем, какую информацию о молекуле они используют. От дескриптора требуется независимость от способа записи молекулы (варианта SMILES, нумерации атомов, а для трёхмерных дескрипторов - ориентации молекулы в пространстве), быстрый расчёт для тысяч и миллионов молекул и связь со свойством, которое предсказывает модель. Функция `Descriptors.CalcMolDescriptors` библиотеки RDKit позволяет быстро вычислять дескрипторы молекулы.

## 1.1 Дескрипторы нулевой размерности

Самый простой способ описать молекулу - записать её брутто-формулу, то есть список элементов с числом атомов каждого элемента. Дескрипторы, которые вычисляют по брутто-формуле, называют 0D-дескрипторами, они не описывают строение молекулы.

| Дескриптор RDKit | Что это |
|---|---|
| `rdMolDescriptors.CalcMolFormula` | брутто-формула |
| `MolWt` | молярная масса $M$, г/моль |
| `HeavyAtomCount` | число атомов $A$ без учёта водорода |
| `NumHeteroatoms` | число атомов, кроме C и H |
| `NumValenceElectrons` | число валентных электронов |

In [ ]:
# Брутто-формула п-хлортолуола и 0D-дескрипторы
mol = Chem.MolFromSmiles("Cc1ccc(Cl)cc1")
symbols = [atom.GetSymbol() for atom in Chem.AddHs(mol).GetAtoms()]   # атомы водорода записаны явно
counts = {x: symbols.count(x) for x in sorted(set(symbols))}          # N_x: сколько раз встречается каждый элемент

Formula = rdMolDescriptors.CalcMolFormula(mol)
A = Descriptors.HeavyAtomCount(mol)
M = Descriptors.MolWt(mol)

print("брутто-формула:", Formula)
print("число атомов каждого типа N_x:", counts)
print(f"A = {A}")
print(f"M = {M:.2f} г/моль")

**Вырожденность.** Дескриптор называют вырожденным, если у разных молекул он принимает одно и то же значение. 0D-дескрипторы сильно вырождены. У изомеров одинаковая брутто-формула, поэтому все 0D-дескрипторы у них совпадают. Но если само свойство у разных молекул почти одинаково, его лучше описывают дескрипторы с такой же вырожденностью. 

In [ ]:
# 0D-дескрипторы и температура кипения
data = pd.DataFrame({
    "вещество": ["этанол", "диметиловый эфир", "гексан", "2-метилпентан", "3-метилпентан", "2,2-диметилбутан"],
    "SMILES":   ["CCO", "COC", "CCCCCC", "CC(C)CCC", "CCC(C)CC", "CC(C)(C)CC"],
    "T_кип":    [78.4, -24.9, 68.8, 60.9, 63.3, 49.8],
}) 

mols = [Chem.MolFromSmiles(s) for s in data["SMILES"]]
data["формула"] = [rdMolDescriptors.CalcMolFormula(mol) for mol in mols]
for name in ["MolWt", "HeavyAtomCount", "NumValenceElectrons", "NumHeteroatoms"]:
    data[name] = [getattr(Descriptors, name)(mol) for mol in mols]
data.drop(columns="SMILES").round(2)

## 1.2 Одномерные дескрипторы

Одномерное представление молекулы - список её структурных фрагментов, то есть функциональных групп, заместителей и атомов вместе с ближайшими соседями. Список может быть неполным, поэтому строение всей молекулы знать не нужно. Дескрипторы, которые вычисляют по такому списку, называют 1D-дескрипторами.

В RDKit фрагмент задают шаблоном SMARTS, а метод `GetSubstructMatches` находит все его вхождения в молекулу:

| SMARTS | фрагмент |
|---|---|
| `[CH3]` | атом C с тремя атомами H |
| `[CH2]` | атом C с двумя атомами H |
| `[CH1]` | атом C с одним атомом H |
| `[CH0;X4]` | атом C без H, с четырьмя соседями |
| `[OX2H]` | группа OH |
| `[OD2]([#6])[#6]` | простой эфир |

Обозначения в SMARTS: `*` - любой атом, `[#8]` - атом с порядковым номером 8 (O), `a` - ароматический атом, `[R]` - атом в цикле, `!H0` - есть хотя бы один атом H; `~` - любая связь, `-` и `=` - одинарная и двойная, `:` - ароматическая, `!:` - неароматическая, `@` - связь в цикле, `!@` - связь вне цикла. «> 1» означает, что ключ срабатывает, если шаблон найден больше одного раза. В шаблонах `H3` означает три атома H у атома, `X4` - четыре соседа с учётом атомов H, `D2` - два соседа без учёта атомов H, `#6` - порядковый номер элемента, а `;` читается как логическое «И».

In [ ]:
# Число фрагментов в молекулах: шаблон SMARTS -> GetSubstructMatches
data = pd.DataFrame({
    "вещество": ["этанол", "диметиловый эфир", "гексан", "2-метилпентан", "3-метилпентан", "2,2-диметилбутан"],
    "SMILES":   ["CCO", "COC", "CCCCCC", "CC(C)CCC", "CCC(C)CC", "CC(C)(C)CC"],
    "T_кип":    [78.4, -24.9, 68.8, 60.9, 63.3, 49.8],
}) 
fragments = {"CH3": "[CH3]",
             "CH2": "[CH2]",
             "CH": "[CH1]", 
             "C": "[CH0;X4]",
             "OH": "[OX2H]", 
             "C-O-C": "[OD2]([#6])[#6]"}

mols = [Chem.MolFromSmiles(s) for s in data["SMILES"]]
for name, smarts in fragments.items():
    pattern = Chem.MolFromSmarts(smarts)
    data["N_" + name] = [len(mol.GetSubstructMatches(pattern)) for mol in mols]   # сколько раз фрагмент встречается
data.drop(columns="SMILES")

У 2-метилпентана и 3-метилпентана набор фрагментов одинаковый и состоит из трёх групп CH3, двух CH2 и одной CH.

**Аддитивные дескрипторы.** Многие свойства приближённо равны сумме вкладов фрагментов:

$$\log P = \sum_k a_k N_k,$$

где $N_k$ - число атомов типа $k$ в молекуле, а $a_k$ - вклад одного такого атома, подобранный регрессией по экспериментальным данным. Для расчёта таких свойств достаточно списка фрагментов, поэтому их тоже относят к 1D-дескрипторам.

| Дескриптор RDKit | Что это |
|---|---|
| `NumHDonors`, `NumHAcceptors` | число доноров и акцепторов водородной связи |
| `MolLogP` | $\log P$ по Wildman-Crippen |
| `MolMR` | молярная рефракция по Wildman-Crippen |
| `TPSA` | полярная площадь поверхности, Å² |
| `LabuteASA` | приближённая площадь поверхности, Å² |

In [ ]:
# Вклады атомов этанола в logP и MR по Wildman-Crippen
mol = Chem.AddHs(Chem.MolFromSmiles("CCO"))                 # атомы водорода записаны явно, у них свои вклады
contribs = rdMolDescriptors._CalcCrippenContribs(mol)       # пары (вклад в logP, вклад в MR) для каждого атома
table = pd.DataFrame({
    "атом":   [atom.GetSymbol() for atom in mol.GetAtoms()],
    "соседи": [" ".join(sorted(n.GetSymbol() for n in atom.GetNeighbors())) for atom in mol.GetAtoms()],
    "a_logP": [c[0] for c in contribs],
    "a_MR":   [c[1] for c in contribs],
})
print(f"сумма вкладов: logP = {table['a_logP'].sum():.4f}, MR = {table['a_MR'].sum():.3f}")
print(f"RDKit:         logP = {Descriptors.MolLogP(mol):.4f}, MR = {Descriptors.MolMR(mol):.3f}")
table

In [ ]:
# 1D-дескрипторы RDKit для тех же шести молекул
data = pd.DataFrame({
    "вещество": ["этанол", "диметиловый эфир", "гексан", "2-метилпентан", "3-метилпентан", "2,2-диметилбутан"],
    "SMILES":   ["CCO", "COC", "CCCCCC", "CC(C)CCC", "CCC(C)CC", "CC(C)(C)CC"],
    "T_кип":    [78.4, -24.9, 68.8, 60.9, 63.3, 49.8],          # °C
})
mols = [Chem.MolFromSmiles(s) for s in data["SMILES"]]
for name in ["fr_Al_OH", "fr_ether", "NumHDonors", "NumHAcceptors", "MolLogP", "MolMR", "TPSA", "LabuteASA"]:
    data[name] = [getattr(Descriptors, name)(mol) for mol in mols]
data.drop(columns="SMILES").round(2)

У этанола есть донор водородной связи, а у диметилового эфира нет. Поэтому у них разные `NumHDonors` и `TPSA`. Аддитивные дескрипторы различают изомеры гексана хуже, чем числа групп CH3, CH2, CH и C из таблицы фрагментов выше. Вырожденность дескриптора зависит от того, какие фрагменты он различает. Чтобы отличить 2-метилпентан от 3-метилпентана, нужно знать, как фрагменты соединены между собой. Это даёт только двумерное представление молекулы.

## 1.3 Двумерные дескрипторы

Двумерное представление молекулы - её граф, в котором вершины соответствуют атомам, а рёбра - химическим связям. В отличие от списка фрагментов, граф показывает, какой атом с каким соединён. 

SMILES - запись того же графа строкой. Дескрипторы, которые вычисляют по графу, называют 2D-дескрипторами. Топологические индексы сводят граф к одному числу, а молекулярные отпечатки описывают его набором окрестностей атомов.

### 1.3.1 Топологические индексы

**Топологический индекс** - число, вычисленное по графу молекулы. При вычислении топологического индекса обычно используют только тяжёлые атомы, а атомы водорода не учитывают. Расстояние $d_{ij}$ между атомами $i$ и $j$ измеряют числом связей на кратчайшем пути между ними. Все расстояния образуют матрицу расстояний $D$.

**Индекс Винера** - сумма расстояний по всем парам атомов. 

$$W = \frac{1}{2}\sum_{i=1}^{N}\sum_{j=1}^{N} d_{ij}$$

Множитель 1/2 убирает двойной счёт, потому что каждая пара входит в сумму дважды, как $d_{ij}$ и $d_{ji}$. Чем сильнее молекула разветвлена, тем она компактнее и тем меньше $W$.

In [ ]:
# Граф 2-метилпентана и матрица расстояний в двух записях SMILES
fig, axes = plt.subplots(2, 2, figsize=(8, 7.2))
for row, smiles in zip(axes, ["CC(C)CCC", "CCCC(C)C"]):
    mol = Chem.MolFromSmiles(smiles)                      # атомы нумеруются в порядке записи SMILES
    G = nx.Graph()                                        # вершины - атомы C, рёбра - связи
    G.add_edges_from((bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()) for bond in mol.GetBonds())
    D = Chem.GetDistanceMatrix(mol)                       # D[i, j] - число связей между атомами i и j
    W = D.sum() / 2                                       # то же даёт nx.wiener_index(G)

    nx.draw(G, pos=nx.kamada_kawai_layout(G), ax=row[0], with_labels=True,
            node_color="#3B6EA8", node_size=500, font_color="white", width=2)
    row[0].set_title(f"{smiles}, W = {W:.0f}")

    row[1].imshow(D, cmap="Blues")
    for i in range(len(D)):
        for j in range(len(D)):
            row[1].text(j, i, f"{D[i, j]:.0f}", ha="center", va="center",
                        color="white" if D[i, j] > 2 else "black")
    row[1].set_xticks(range(len(D)))
    row[1].set_yticks(range(len(D)))
    row[1].set_title("матрица расстояний D")
plt.tight_layout()
plt.show()

### 1.3.2 Молекулярные отпечатки

**Молекулярный отпечаток** (fingerprint) - строка из нулей и единиц. Каждая позиция показывает, есть ли в молекуле определённый фрагмент, 1 означает «есть», а 0 - «нет». Фрагменты выбирают одним из двух способов.

Метод расчёта молекулярных отпечатков **MACCS** (Molecular ACCess System), разработанный компанией MDL (BIOVIA), представляет собой заранее составленный список из 166 ключей. Ключ - это шаблон SMARTS с порогом. Если шаблон встречается в молекуле большее число раз, чем указано в пороге, на месте этого ключа в векторе ставится 1. В векторе 167 позиций, потому что позиция 0 не используется.

In [ ]:
# Ключи MACCS аспирина: расчёт, визуализация и таблица
mol = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")             # аспирин
fp = MACCSkeys.GenMACCSKeys(mol)                              # строка из 167 позиций
keys = list(fp.GetOnBits())                                   # номера ключей с ответом «да»

# Какие атомы нашёл каждый ключ и сколько раз
rows, highlights, legends = [], [], []
for k in keys:
    smarts, threshold = MACCSkeys.smartsPatts[k]              # «да», если совпадений с шаблоном больше порога
    if smarts == "?":                                         # у ключей 1, 125 и 166 нет шаблона SMARTS: 125 и 166 RDKit проверяет отдельным кодом, ключ 1 не реализован
        continue
    matches = mol.GetSubstructMatches(Chem.MolFromSmarts(smarts))
    highlights.append(sorted({atom for match in matches for atom in match}))
    legends.append(f"key {k}: {smarts}" + (f" > {threshold}" if threshold else "")
                   + f"\nfound: {len(matches)}")              # сколько раз шаблон нашёлся
    rows.append((k, smarts, threshold, len(matches)))

opts = Draw.MolDrawOptions()
opts.legendFontSize = 22                                      # размер шрифта подписей
opts.legendFraction = 0.25                                    # доля высоты картинки под подпись
display(Draw.MolsToGridImage([mol] * len(highlights), highlightAtomLists=highlights, legends=legends,
                             molsPerRow=5, subImgSize=(300, 250), drawOptions=opts))

maccs = pd.DataFrame(rows, columns=["ключ", "SMARTS", "порог", "найдено"])   # «да», если найдено > порог
maccs

| Ключ | SMARTS | Что означает |
|---|---|---|
| 89 | `[#8]~*~*~*~[#8]` | два атома O, между которыми три атома |
| 113 | `[#8]!:*:*` | атом O, присоединённый к ароматическому кольцу |
| 123 | `[#8]~[#6]~[#8]` | атом C с двумя атомами O |
| 126 | `*!@[#8]!@*` | атом O между двумя атомами, обе связи вне цикла (эфирный O) |
| 127 | `*@*!@[#8]` > 1 | атом O при цикле, шаблон найден больше одного раза |
| 136 | `[#8]=*` > 1 | больше одного атома O с двойной связью |
| 139 | `[O;!H0]` | группа OH |
| 140 | `[#8]` > 3 | больше трёх атомов O |
| 143 | `*@*!@[#8]` | атом O при цикле |
| 144 | `*!:*:*!:*` | заместители у двух соседних ароматических атомов |
| 146 | `[#8]` > 2 | больше двух атомов O |
| 150 | `*!@*@*!@*` | заместители у двух соседних атомов цикла |
| 152 | `[#8]~[#6](~[#6])~[#6]` | атом C, связанный с O и двумя атомами C |
| 154 | `[#6]=[#8]` | группа C=O |
| 157 | `[#6]-[#8]` | одинарная связь C-O |
| 159 | `[#8]` > 1 | больше одного атома O |
| 160 | `[C;H3,H4]` | группа CH3 |
| 162 | `a` | ароматический атом |
| 163 | `*1~*~*~*~*~*~1` | шестичленный цикл |
| 164 | `[#8]` | атом O |
| 165 | `[R]` | атом в цикле |

Отпечаток **Моргана**, или ECFP (Extended-Connectivity FingerPrint), описывает молекулу набором атом-центрированных фрагментов. Готового словаря фрагментов, как у MACCS, здесь нет, фрагменты получают из самой молекулы вариантом алгоритма расширенной связности Моргана.

Отпечаток строится по шагам, шаг называют радиусом $r$.

**$r = 0$.** Для каждого атома составляют список его свойств, например порядковый номер элемента, число соседей, число соседних атомов водорода, заряд, поправка на изотоп и так далее. Далее полученный вектор, например `(6, 4, 3, 0, 0)` передаётся в хэш функцию, из которая возвращает одно число (**номер фрагмента**) от 0 до $2^{32} - 1$, например `2246728737`. Одинаковые списки свойств всегда дают одно и то же число. Далее для данного фрагмента можно найти его положение в векторе, получив остаток от деления номера фрагмента на длину вектора, например при длине вектора `2048` остаток от деления числа `2246728737` является `1057`. Значит именно положение `1057` в векторе отвечает за наличие этого фрагмента. Такой переход от номера фрагмента к позиции называют свёрткой. Номеров намного больше, чем позиций, поэтому два разных фрагмента иногда попадают в одну позицию.

**$r = 1$.** Для каждого атома составляют новый список уже не из свойств, а из чисел предыдущего шага ($r=0$). В него входят номер самого атома и для каждого соседа пара из типа связи и номера соседа, причём пары идут по возрастанию. Например, если атом на предыдущем шаге получил номер `2246728737`, а его единственный сосед, соединённый одинарной связью (тип 1), получил номер `2245384272`, то в хеш-функцию передают список `(2246728737, (1, 2245384272))`. Расчёт хеша начинается с номера шага (здесь 0), и в результате получается новый номер фрагмента `3542456614` с положением в векторе `294`.

**$r = 2$.** То же самое, но список собирают из чисел шага $r = 1$, тем самым учитывая всё более дальние атомы. Если фрагмент этого шага охватывает те же связи, что уже найденный фрагмент, RDKit его отбрасывает, поэтому у этанола новых фрагментов при $r = 2$ нет.

In [ ]:
# Отпечаток Моргана этанола: расчёт и ненулевые позиции вектора
mol = Chem.MolFromSmiles("CCO")                               # этанол
size = 2048                                                   # длина вектора
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=size)
fp = gen.GetFingerprint(mol)                                  # вектор из 0 и 1

# Таблица ненулевых значений вектора
x = np.array(fp)                                              # вектор как массив numpy
positions = np.flatnonzero(x)                                 # номера позиций, где значение не равно 0
print(f"длина вектора: {len(x)}, ненулевых значений: {len(positions)}")
pd.DataFrame({"позиция": positions, "значение": x[positions]})

In [ ]:
# Номера фрагментов до свёртки и атомы, которые их дали
ao = rdFingerprintGenerator.AdditionalOutput()
ao.AllocateBitInfoMap()
gen.GetSparseCountFingerprint(mol, additionalOutput=ao)
frag_info = ao.GetBitInfoMap()

# SMILES фрагмента: центральный атом и все атомы не дальше radius связей от него
def fragment_smiles(mol, atom, radius):
    bonds = list(Chem.FindAtomEnvironmentOfRadiusN(mol, radius, atom))
    atoms = {atom}
    for b in bonds:
        atoms.update((mol.GetBondWithIdx(b).GetBeginAtomIdx(), mol.GetBondWithIdx(b).GetEndAtomIdx()))
    return Chem.MolFragmentToSmiles(mol, atomsToUse=list(atoms), bondsToUse=bonds, allHsExplicit=True)

# Таблица: один фрагмент - одна строка
rows = []
for frag_id, places in frag_info.items():
    atom, radius = places[0]                                  # первое место, где нашёлся фрагмент
    rows.append({"атом": atom, "радиус": radius, "фрагмент": fragment_smiles(mol, atom, radius),
                 "найдено": len(places), "номер фрагмента": frag_id, "позиция": frag_id % size})
morgan = pd.DataFrame(rows).sort_values(["радиус", "атом"], ignore_index=True)

# Рисунок: один фрагмент - одна картинка
opts = Draw.MolDrawOptions()
opts.legendFontSize = 16                                      # размер шрифта подписей
opts.legendFraction = 0.25                                    # доля высоты картинки под подпись
legends = [f"{smiles}\nposition {position}" for smiles, position in zip(morgan["фрагмент"], morgan["позиция"])]
display(Draw.DrawMorganBits([(mol, frag_id, frag_info) for frag_id in morgan["номер фрагмента"]],
                            legends=legends, molsPerRow=3, subImgSize=(200, 170), drawOptions=opts))
morgan

In [ ]:
# Номера фрагментов этанола вручную: тот же расчёт, что делает RDKit
def hash_list(values, seed=0):                 # хеш-функция: список целых чисел -> одно 32-битное число
    for value in values:
        if isinstance(value, tuple):           # пара (тип связи, номер соседа) сначала сама превращается в число
            value = hash_list(value)
        seed = seed ^ ((value + 0x9e3779b9 + (seed << 6) + (seed >> 2)) % 2**32)
    return seed

mol = Chem.MolFromSmiles("CCO")
periodic_table = Chem.GetPeriodicTable()       # таблица Менделеева: массы элементов для поправки на изотоп

# Радиус 0: хеш списка свойств атома
props = [[atom.GetAtomicNum(),                                                         # порядковый номер элемента
          atom.GetTotalDegree(),                                                       # число соседей, включая атомы H
          atom.GetTotalNumHs(),                                                        # число атомов H
          atom.GetFormalCharge(),                                                      # заряд
          int(atom.GetMass() - periodic_table.GetAtomicWeight(atom.GetAtomicNum()))]   # поправка на изотоп
         + [1] * atom.IsInRing()                                                       # 1, если атом в цикле
         for atom in mol.GetAtoms()]
r0 = [hash_list(p) for p in props]

# Радиус 1: номер самого атома и пары (тип связи, номер соседа) по возрастанию; расчёт хеша начинается с номера шага 0
r1 = [hash_list([r0[atom.GetIdx()]]
                + sorted((int(bond.GetBondType()), r0[bond.GetOtherAtomIdx(atom.GetIdx())]) for bond in atom.GetBonds()), seed=0)
      for atom in mol.GetAtoms()]

print("свойства атомов:", props)
print("радиус 0, номера фрагментов:", r0, "позиции:", [n % 2048 for n in r0])
print("радиус 1, номера фрагментов:", r1, "позиции:", [n % 2048 for n in r1])

### 1.3.3 Коэффициент Танимото

**Сходство молекул.** Два отпечатка сравнивают с помощью коэффициента Танимото - доли общих единиц среди всех единиц обоих векторов:

$$T_{AB} = \frac{N_{AB}}{N_A + N_B - N_{AB}}$$

$N_A$ и $N_B$ - числа единиц в отпечатках молекул $A$ и $B$, $N_{AB}$ - число единиц, общих для обоих. $T = 1$ у одинаковых отпечатков, $T = 0$, если общих единиц нет. 

In [ ]:
# Структуры четырёх пар лекарств
drugs = {
    "кодеин":       "CN1CC[C@]23c4c5ccc(OC)c4O[C@H]2[C@@H](O)C=C[C@H]3[C@H]1C5",
    "морфин":       "CN1CC[C@]23c4c5ccc(O)c4O[C@H]2[C@@H](O)C=C[C@H]3[C@H]1C5",
    "гидроксизин":  "OCCOCCN1CCN(CC1)C(c1ccccc1)c1ccc(Cl)cc1",
    "цетиризин":    "OC(=O)COCCN1CCN(CC1)C(c1ccccc1)c1ccc(Cl)cc1",
    "лоратадин":    "CCOC(=O)N1CCC(=C2c3ccc(Cl)cc3CCc3cccnc32)CC1",
    "дезлоратадин": "Clc1ccc2c(c1)CCc1cccnc1C2=C1CCNCC1",
    "ибупрофен":    "CC(C)Cc1ccc(cc1)C(C)C(=O)O",
    "напроксен":    "COc1ccc2cc(ccc2c1)C(C)C(=O)O",
}

fig, axes = plt.subplots(2, 4, figsize=(12.5, 5))
for ax, (name, smiles) in zip(axes.flat, drugs.items()):
    ax.imshow(Draw.MolToImage(Chem.MolFromSmiles(smiles), size=(400, 300)))
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Матрица сходства Танимото по отпечаткам Моргана
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fps = [gen.GetFingerprint(Chem.MolFromSmiles(s)) for s in drugs.values()]
sim = np.array([DataStructs.BulkTanimotoSimilarity(fp, fps) for fp in fps])

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(sim, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(drugs)), list(drugs), rotation=90)
ax.set_yticks(range(len(drugs)), list(drugs))
for i in range(len(drugs)):
    for j in range(len(drugs)):
        ax.text(j, i, f"{sim[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if sim[i, j] > 0.6 else "black")
fig.colorbar(im, ax=ax, label="коэффициент Танимото")
plt.tight_layout()
plt.show()

### 1.3.4 Поиск по базам данных

In [ ]:
# Локальная база соединений: 130 ингибиторов трипсина
base = pd.read_csv("trypsin_ligands.csv")
base["mol"] = [Chem.MolFromSmiles(s) for s in base["SMILES"]]
base.head()


In [ ]:
# Какие функциональные группы чаще всего встречаются в базе
rows = []
with open(Fragments.defaultPatternFileName) as file:  # файл RDKit со списком групп: имя, описание, SMARTS
    for line in file:
        parts = line.rstrip("\n").split("\t")
        if not line.startswith("#") and len(parts) >= 3:
            name = parts[0].replace("=", "_").replace("-", "_")   # так RDKit называет дескриптор
            rows.append({"дескриптор": name, "описание": parts[1].strip('"'), "SMARTS": parts[2]})
groups = pd.DataFrame(rows)
groups["молекул"] = [sum(getattr(Descriptors, name)(mol) > 0 for mol in base["mol"]) for name in groups["дескриптор"]]
print(f"шаблонов: {len(groups)}, молекул в базе: {len(base)}")
groups.sort_values("молекул", ascending=False).head(10)

In [ ]:
# Поиск по подструктуре: молекулы с амидиновой группой при ароматическом кольце
amidine = Chem.MolFromSmarts("C(=N)(-N)-[!#7]")           # фрагмент бензамидина: c-C(=N)-N
found = base[[mol.HasSubstructMatch(amidine) for mol in base["mol"]]]
print(f"ариламидинов: {len(found)} из {len(base)}")

sample = found.sample(8, random_state=0)                      # восемь случайных примеров
display(Draw.MolsToGridImage(list(sample["mol"]), molsPerRow=4, subImgSize=(250, 200),
                             legends=[f"{i}, pKi = {p:.2f}" for i, p in zip(sample["ChEMBL_ID"], sample["pKi"])],
                             highlightAtomLists=[[a for match in mol.GetSubstructMatches(amidine) for a in match]
                                                 for mol in sample["mol"]]))
found[["ChEMBL_ID", "SMILES", "pKi"]].head()

In [ ]:
# Поиск по сходству в PubChem: молекулы, похожие на 2-нафтамидин
import urllib.parse

smiles = "NC(=N)c1ccc2ccccc2c1"                               # 2-нафтамидин
url = ("https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/fastsimilarity_2d/smiles/"
       + urllib.parse.quote(smiles, safe="")                  # SMILES в виде, допустимом в адресе
       + "/property/Title,SMILES/CSV?Threshold=90&MaxRecords=8")   # сходство по PubChem не ниже 90 %, не больше 8 молекул
hits = pd.read_csv(url)                                       # ответ PubChem: CID, название, SMILES

# Пересчитываем сходство по отпечатку Моргана и сортируем
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
query_fp = gen.GetFingerprint(Chem.MolFromSmiles(smiles))
hits["mol"] = [Chem.MolFromSmiles(s) for s in hits["SMILES"]]
hits["T"] = [DataStructs.TanimotoSimilarity(query_fp, gen.GetFingerprint(mol)) for mol in hits["mol"]]
hits = hits.sort_values("T", ascending=False, ignore_index=True)

display(Draw.MolsToGridImage(list(hits["mol"][:8]), molsPerRow=4, subImgSize=(250, 200),
                             legends=[f"CID {c}, T = {t:.2f}" for c, t in zip(hits["CID"][:8], hits["T"][:8])]))
hits[["CID", "Title", "T"]].round(2)

## 1.4 Трёхмерные дескрипторы

Молекулу можно представлять в трёхмерном пространстве, где у каждого атома есть координаты $(x, y, z)$. Дескрипторы, вычисляемые по координатам, называют 3D-дескрипторами. Основная задача трёхмерных дескрипторов - разичать конформации одной молекулы. В отличие от двумерного графа, здесь нужно учитывать систему координат. При повороте и сдвиге молекулы как целого координаты атомов меняются, а дескриптор меняться не должен. 

### 1.4.1 Матрица геометрии

**Матрица геометрии** $G$ - аналог матрицы расстояний $D$ для двумерного графа. В матрице $D$ находится число связей между атомами $i$ и $j$, а в матрице $G$ - геометрическое расстояние $r_{ij}$ между атомами в пространстве. Полусумма элементов матрицы $G$ даёт **трёхмерный индекс Винера**:

$$^{3D}W = \frac{1}{2}\sum_{i}\sum_{j} r_{ij}$$

В RDKit матрицу $G$ возвращает функция `Chem.Get3DDistanceMatrix`. Сравним две геометрии бутана, гош-конформер и анти-конформер.

In [ ]:
# 3D-структуры двух конформеров бутана в окне WebMol
gauche = Chem.AddHs(Chem.MolFromSmiles("CCCC"))
AllChem.EmbedMolecule(gauche, randomSeed=42)                  # 3D-геометрия методом ETKDG
AllChem.MMFFOptimizeMolecule(gauche)                          # оптимизация MMFF: гош-конформер
anti = Chem.Mol(gauche)                                       # копия молекулы
rdMolTransforms.SetDihedralDeg(anti.GetConformer(), 0, 1, 2, 3, 180.0)   # поворот вокруг центральной связи C-C

In [ ]:
md_tools.show({"gauche": gauche, "anti": anti}, cmd="hide everything; show sticks")

In [ ]:
# Матрицы геометрии двух конформеров бутана и трёхмерный индекс Винера
fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
for ax, (name, mol) in zip(axes, [("гош-конформер", gauche), ("анти-конформер", anti)]):
    carbons = Chem.RemoveHs(mol)                              # оставляем атомы C, как в графе
    G = Chem.Get3DDistanceMatrix(carbons)                     # G[i, j] - расстояние между атомами i и j, Å
    W_3d = G.sum() / 2                                        # трёхмерный индекс Винера, Å
    W_2d = Chem.GetDistanceMatrix(carbons).sum() / 2          # обычный индекс Винера по числу связей
    ax.imshow(G, cmap="Blues", vmin=0, vmax=4)
    for i in range(len(G)):
        for j in range(len(G)):
            ax.text(j, i, f"{G[i, j]:.2f}", ha="center", va="center", color="white" if G[i, j] > 2.5 else "black")
    ax.set_xticks(range(len(G)))
    ax.set_yticks(range(len(G)))
    ax.set_title(f"{name}\n$^{{3D}}W$ = {W_3d:.2f} Å, $^{{2D}}W$ = {W_2d:.0f}")
plt.tight_layout()
plt.show()

### 1.4.2 Форма молекулы

Форму молекулы можно описать через распределение масс атомов. Момент инерции молекулы относительно оси равен $I = \sum_i m_i d_i^2$, где $m_i$ - масса атома, $d_i$ - его расстояние до оси. У любой молекулы есть три взаимно перпендикулярные **главные оси**, причём относительно одной из них момент инерции наименьший, а относительно другой - наибольший. Моменты относительно главных осей называют **главными моментами инерции** $I_1 \le I_2 \le I_3$. Главные оси связаны с самой молекулой и поворачиваются вместе с ней, поэтому главные моменты не зависят от системы координат.

In [ ]:
# Три опорные молекулы: стержень, диск и шар
def embed(smiles, n_confs=1):
    """3D-геометрии молекулы: ETKDG + оптимизация MMFF; возвращает молекулу и энергии конформеров, кДж/моль"""
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, randomSeed=42)
    result = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=2000)
    return mol, np.array([energy for not_converged, energy in result]) * 4.184

rod, _ = embed("C#CC#C")                                      # диацетилен
disc, _ = embed("c1ccccc1")                                   # бензол
ball, _ = embed("C1C2CC3CC1CC(C2)C3")                         # адамантан
for k, mol in enumerate([rod, disc, ball]):                   # раздвигаем молекулы на 8 Å по оси x, чтобы не накладывались
    conf = mol.GetConformer()
    conf.SetPositions(conf.GetPositions() + np.array([8.0 * k, 0.0, 0.0]))

In [ ]:
md_tools.show({"rod": rod, "disc": disc, "ball": ball}, cmd="hide everything; show sticks", height=400)

Из главных моментов получают дескрипторы формы, которые не зависят от размера молекулы:

- отношения $\mathrm{NPR1} = I_1 / I_3$ и $\mathrm{NPR2} = I_2 / I_3$ (NPR - normalized principal moments ratio). Точка (NPR1, NPR2) лежит в треугольнике с вершинами «стержень» (0, 1), «диск» (0.5, 0.5) и «шар» (1, 1);
- **асферичность** - одно число, которое показывает отклонение формы от шара и равно 0 у шара, 0.25 у диска и 1 у стержня.

Размер молекулы описывает **радиус гирации** $R_g$ - среднеквадратичное расстояние атомов от центра масс:

$$R_g = \sqrt{\frac{\sum_i m_i r_i^2}{\sum_i m_i}}$$

где $r_i$ - расстояние от атома до центра масс. У гибкой молекулы все эти величины зависят от конформации. 

In [ ]:
# Главные моменты инерции и дескрипторы формы трёх молекул
rows = []
for name, mol in [("диацетилен", rod), ("бензол", disc), ("адамантан", ball)]:
    rows.append({"молекула":      name,
                 "I1":            rdMolDescriptors.CalcPMI1(mol),              # главные моменты инерции, а.е.м.·Å²
                 "I2":            rdMolDescriptors.CalcPMI2(mol),
                 "I3":            rdMolDescriptors.CalcPMI3(mol),
                 "NPR1":          rdMolDescriptors.CalcNPR1(mol),              # I1 / I3
                 "NPR2":          rdMolDescriptors.CalcNPR2(mol),              # I2 / I3
                 "асферичность":  rdMolDescriptors.CalcAsphericity(mol),
                 "Rg, Å":         rdMolDescriptors.CalcRadiusOfGyration(mol)})
pd.DataFrame(rows).round(2)

In [ ]:
# 100 конформеров декана: все геометрии - состояния одного объекта в WebMol
decane, E = embed("CCCCCCCCCC", n_confs=100)
dE = E - E.min()                                              # энергия относительно самого низкого конформера, кДж/моль
print(f"конформеров: {decane.GetNumConformers()}, энергия от 0 до {dE.max():.1f} кДж/моль")
md_tools.show({"decane": decane}, cmd="hide everything; show sticks", height=400)

In [ ]:
# Дескрипторы формы каждого конформера декана
conformers = pd.DataFrame({
    "dE":           dE,
    "NPR1":         [rdMolDescriptors.CalcNPR1(decane, confId=c.GetId()) for c in decane.GetConformers()],
    "NPR2":         [rdMolDescriptors.CalcNPR2(decane, confId=c.GetId()) for c in decane.GetConformers()],
    "асферичность": [rdMolDescriptors.CalcAsphericity(decane, confId=c.GetId()) for c in decane.GetConformers()],
    "Rg":           [rdMolDescriptors.CalcRadiusOfGyration(decane, confId=c.GetId()) for c in decane.GetConformers()],
})

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

ax = axes[0]                                                  # треугольник формы
ax.plot([0, 0.5, 1, 0], [1, 0.5, 1, 1], color="black", lw=0.8)
for name, mol, offset in [("диацетилен", rod, (8, 6)), ("бензол", disc, (8, -4)), ("адамантан", ball, (-70, 6))]:
    npr1, npr2 = rdMolDescriptors.CalcNPR1(mol), rdMolDescriptors.CalcNPR2(mol)
    ax.plot(npr1, npr2, "s", color="#DC602E", ms=7)
    ax.annotate(name, (npr1, npr2), textcoords="offset points", xytext=offset)
points = ax.scatter(conformers["NPR1"], conformers["NPR2"], c=conformers["dE"], cmap="viridis", s=15)
fig.colorbar(points, ax=ax, label="ΔE, кДж/моль")
ax.set_ylim(0.45, 1.08)
ax.set_xlabel("NPR1")
ax.set_ylabel("NPR2")
ax.set_title("форма конформеров декана")

ax = axes[1]                                                  # размер и энергия
ax.scatter(conformers["Rg"], conformers["dE"], color="#3B6EA8", s=15)
ax.set_xlabel("радиус гирации $R_g$, Å")
ax.set_ylabel("ΔE, кДж/моль")
ax.set_title("размер и энергия")
plt.tight_layout()
plt.show()

В этом разделе показаны только простейшие 3D-дескрипторы. Остальные, доступные для расчёта в RDKit, представлены в ячейке ниже.

In [ ]:
# 3D-дескрипторы RDKit для 100 конформеров декана
conf_ids = [conf.GetId() for conf in decane.GetConformers()]  # номера конформеров
radii = rdFreeSASA.classifyAtoms(decane)                      # радиусы атомов для расчёта поверхности

# Скалярные дескрипторы: одно число на конформер
scalars = pd.DataFrame([Descriptors3D.CalcMolDescriptors3D(decane, confId=c) for c in conf_ids])   # 11 дескрипторов размера и формы
scalars["SASA"] = [rdFreeSASA.CalcSASA(decane, radii, confIdx=c) for c in conf_ids]                # площадь поверхности, доступной растворителю, Å²
scalars["Volume"] = [AllChem.ComputeMolVolume(decane, confId=c) for c in conf_ids]                 # объём молекулы, Å³

# Векторные дескрипторы: набор чисел на конформер
vectors = {
    "RDF":        rdMolDescriptors.CalcRDF,           # радиальная функция распределения: сколько пар атомов на каждом расстоянии
    "MORSE":      rdMolDescriptors.CalcMORSE,         # 3D-MoRSE: расчётная картина рассеяния электронов на атомах молекулы
    "AUTOCORR3D": rdMolDescriptors.CalcAUTOCORR3D,    # 3D-автокорреляция: как связаны свойства атомов на разных расстояниях
    "WHIM":       rdMolDescriptors.CalcWHIM,          # размер, форма и симметрия вдоль главных осей молекулы
    "GETAWAY":    rdMolDescriptors.CalcGETAWAY,       # геометрия вместе с графом: влияние каждого атома на форму молекулы
    "USR":        rdMolDescriptors.GetUSR,            # форма: распределение расстояний атомов до четырёх опорных точек
    "USRCAT":     rdMolDescriptors.GetUSRCAT,         # то же отдельно для гидрофобных, ароматических атомов, доноров и акцепторов
}
tables = [scalars]
for name, function in vectors.items():
    values = np.array([function(decane, confId=c) for c in conf_ids])
    tables.append(pd.DataFrame(values, columns=[f"{name}_{k + 1}" for k in range(values.shape[1])]))

X_3d = pd.concat(tables, axis=1)                              # матрица признаков: строки - конформеры, столбцы - дескрипторы
print(f"конформеров: {X_3d.shape[0]}, 3D-дескрипторов: {X_3d.shape[1]}")
X_3d

## 1.5 Многомерные дескрипторы

Дескрипторы предыдущих разделов описывали саму молекулу. Четырёхмерное представление описывает пространство вокруг неё. Молекулу помещают в регулярную сетку и в каждый узел по очереди ставят пробную частицу - **зонд**. Энергии взаимодействия зонда с молекулой во всех узлах образуют **поле молекулярного взаимодействия**. Дескриптором молекулы служит набор значений поля во всех узлах сетки.

In [ ]:
# Шаг 1. Молекула: геометрия, заряды атомов и ориентация по главным осям
def embed_charged(smiles):
    """3D-геометрия молекулы: ETKDG + оптимизация MMFF + частичные заряды атомов"""
    molecule = Chem.AddHs(Chem.MolFromSmiles(smiles))
    AllChem.EmbedMolecule(molecule, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(molecule, maxIters=2000)
    AllChem.ComputeGasteigerCharges(molecule)                 # заряды нужны для электростатического поля
    return molecule

mol = embed_charged("NC(=[NH2+])c1ccccc1")                    # катион бензамидиния
Chem.rdMolTransforms.CanonicalizeConformer(mol.GetConformer())   # центр молекулы - в начало координат, главные оси - вдоль x, y, z

In [ ]:
# Шаг 2. Сетка: узлы с шагом 2 Å вокруг молекулы
grid = Chem.rdMIF.ConstructGrid(mol, margin=4.0, spacing=2.0)                   # шаг 2 Å, запас 4 Å вокруг молекулы
xyz = np.array([list(grid.GetGridPointLoc(i)) for i in range(grid.GetSize())])  # координаты узлов, Å
print(f"узлов сетки: {grid.GetNumX()} x {grid.GetNumY()} x {grid.GetNumZ()} = {grid.GetSize()}")

nodes = Chem.RWMol()                                          # узлы для WebMol: каждый узел записан атомом He
for _ in xyz:
    nodes.AddAtom(Chem.Atom("He"))
conf = Chem.Conformer(len(xyz))
conf.SetPositions(xyz)
nodes.AddConformer(conf)
scene = Chem.CombineMols(mol, nodes.GetMol())                 # молекула и узлы сетки в одной структуре

In [ ]:
md_tools.show({"grid": scene}, height=450,
              cmd="hide everything; show sticks, not elem He; show nb_spheres, elem He; color grey70, elem He")

In [ ]:
# Шаг 3. Поля: в каждом узле - энергия взаимодействия молекулы с зондом
def fields(grid, molecule):
    """Значения двух полей во всех узлах сетки, кДж/моль"""
    Chem.rdMIF.CalculateDescriptors(grid, Chem.rdMIF.Coulomb(molecule, probeCharge=1.0))       # зонд - заряд +1
    electro = np.array(grid.GetOccupancyVect())
    Chem.rdMIF.CalculateDescriptors(grid, Chem.rdMIF.MMFFVdWaals(molecule, probeAtomType=1))   # зонд - атом C (sp3)
    steric = (np.array(grid.GetOccupancyVect()) * 4.184).clip(max=50)                          # ккал/моль -> кДж/моль; отталкивание c отсечкой
    return pd.DataFrame({"electro": electro, "steric": steric})

table = pd.concat([pd.DataFrame(xyz, columns=["x", "y", "z"]), fields(grid, mol)], axis=1)     # строка - узел сетки
table

In [ ]:
# Электростатическое поле в WebMol: цвет узла - энергия зонда +1
b = [0.0] * mol.GetNumAtoms() + list(table["electro"])        # число на каждый атом сцены: у узлов - значение поля
md_tools.show({"electro": scene}, b=b, height=450,
              cmd="hide everything; show sticks, not elem He; show nb_spheres, elem He; spectrum b, blue_white_red, elem He")

In [ ]:
# Стерическое поле в WebMol: цвет узла - энергия зонда C
b = [0.0] * mol.GetNumAtoms() + list(table["steric"].clip(-3, 3))
md_tools.show({"steric": scene}, b=b, height=450,
              cmd="hide everything; show sticks, not elem He; show nb_spheres, elem He; spectrum b, blue_white_red, elem He, -3, 3")

In [ ]:
# Шаг 4. Дескриптор молекулы: значения обоих полей во всех узлах подряд
x = np.concatenate([table["electro"], table["steric"]])
print(f"узлов: {len(table)}, полей: 2, длина вектора-дескриптора: {len(x)}")

In [ ]:
# Шаг 5. Набор молекул: 100 лигандов трипсина, совмещённых по общему фрагменту
base = pd.read_csv("trypsin_ligands.csv")
core = Chem.MolFromSmarts("c(c)(c)[CX3](=[NX2])[NX3]")        # общий фрагмент: амидиновая группа и три атома кольца
ligands = base[[Chem.MolFromSmiles(s).HasSubstructMatch(core) for s in base["SMILES"]]].reset_index(drop=True)

reference = embed_charged("NC(=N)c1ccccc1")                   # бензамидин - образец, на который накладываем остальные
Chem.rdMolTransforms.CanonicalizeConformer(reference.GetConformer())
ref_atoms = reference.GetSubstructMatch(core)

mols = []
for smiles in ligands["SMILES"]:
    ligand = embed_charged(smiles)
    matches = ligand.GetSubstructMatches(core, uniquify=False)   # все способы наложить общий фрагмент на образец
    rmsd = [AllChem.AlignMol(ligand, reference, atomMap=list(zip(match, ref_atoms))) for match in matches]
    AllChem.AlignMol(ligand, reference, atomMap=list(zip(matches[int(np.argmin(rmsd))], ref_atoms)))   # лучшее наложение
    mols.append(ligand)

aligned = mols[0]
for ligand in mols[1:]:
    aligned = Chem.CombineMols(aligned, ligand)               # все совмещённые лиганды в одной структуре

In [ ]:
# Шаг 6. Матрица признаков: поля каждого лиганда в узлах общей сетки
common_grid = Chem.rdMIF.ConstructGrid(aligned, margin=3.0, spacing=2.0)   # одна сетка вокруг всех лигандов
n = common_grid.GetSize()

rows = []
for ligand in mols:
    values = fields(common_grid, ligand)                      # те же два поля, что на шаге 3
    rows.append(np.concatenate([values["electro"], values["steric"]]))

X_4d = pd.DataFrame(rows, index=ligands["ChEMBL_ID"],
                    columns=[f"E_{k}" for k in range(n)] + [f"S_{k}" for k in range(n)])   # E - электростатическое, S - стерическое
y = ligands["pKi"].to_numpy()
print(f"узлов общей сетки: {n}, матрица X: {X_4d.shape[0]} молекул x {X_4d.shape[1]} дескрипторов")
X_4d

Поле зависит от того, как молекула повёрнута относительно сетки. Поэтому молекулы сначала совмещают по общему фрагменту и только потом считают поля на одной общей сетке, и тогда столбец матрицы соответствует одной и той же точке пространства у всех молекул. Строка матрицы - одна молекула, столбец - значение поля в одном узле. Всего в матрице 3276 столбцов электростатического поля и 3276 столбцов стерического. Дескрипторов в 65 раз больше, чем молекул, поэтому для таких матриц применяют методы, которые сначала сжимают матрицу до нескольких новых признаков, или модели с регуляризацией.

# 🌳 Глава 2. Ансамблевые методы

**Ансамбль** - модель, которая объединяет ответы нескольких моделей.
Есть три основных способа построить ансамбль:
- **Беггинг (bagging)** - обучаем несколько моделей независимо друг от друга на немного разных данных, затем усредняем их ответы. К нему относится метод случайного леса (Random Forest, RF).
- **Бустинг (boosting)** - обучаем модели по очереди таким образом, чтобы каждая следующая модель исправляла ошибки предыдущих. К нему относят алгоритм AdaBoost и градиентный бустинг, который реализован в библиотеках XGBoost, LightGBM и CatBoost.
- **Стекинг (stacking)** - обучаем несколько разных моделей независимо и добавляем отдельную метамодель, которая учится правильно объединять их ответы. 

In [ ]:
# Пример из занятия 10: поверхность потенциальной энергии
def energy(r1, r2, D=460.0, a=2.3, r0=0.96, k12=-100.0):   # модельные параметры: кДж/моль, 1/Å, Å, кДж/(моль·Å²)
    """Энергия, кДж/моль: два потенциала Морзе и взаимодействие связей"""
    morse_1 = D * (1 - np.exp(-a * (r1 - r0))) ** 2  # потенциал Морзе первой связи
    morse_2 = D * (1 - np.exp(-a * (r2 - r0))) ** 2  # потенциал Морзе второй связи
    return morse_1 + morse_2 + k12 * (r1 - r0) * (r2 - r0)

rng = np.random.default_rng(0)
noise = 15.0                                         # шум в энергиях, кДж/моль: погрешность «измерения»
X_train = rng.uniform(0.75, 1.8, size=(400, 2))      # признаки: длины связей r1 и r2, Å
y_train = energy(X_train[:, 0], X_train[:, 1]) + rng.normal(0, noise, size=400)
X_test = rng.uniform(0.75, 1.8, size=(2000, 2))
y_test = energy(X_test[:, 0], X_test[:, 1]) + rng.normal(0, noise, size=2000)
print(f"обучающая выборка: {X_train.shape}, тестовая: {X_test.shape}")

# Точная поверхность и точки обучающей выборки
r_grid = np.linspace(0.75, 1.8, 100)
R1, R2 = np.meshgrid(r_grid, r_grid)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
image = ax.contourf(R1, R2, energy(R1, R2), levels=20, cmap="RdYlBu_r")
ax.scatter(X_train[:, 0], X_train[:, 1], color="black", s=3)
fig.colorbar(image, ax=ax, label="E, кДж/моль")
ax.set_xlabel("r$_1$, Å")
ax.set_ylabel("r$_2$, Å")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()


## 2.1 Дерево решений

**Дерево решений** (decision tree) - модель, которая при обучении подбирает условия вида `признак <= или > порога` и шаг за шагом делит по ним объекты обучающей выборки на группы. Конечные группы называют листьями. Каждому листу присвоено своё предсказание - среднее значение $y$ его объектов. Признак и порог каждого условия выбирают так, чтобы сумма квадратов отклонений $y$ от среднего в двух получившихся группах была наименьшей. Условия и листья вместе называют узлами дерева, а наибольшее число условий на пути до листа - его глубиной.

In [ ]:
# Простой способ построить дерево решений - scikit-learn
tree_sk = sklearn.tree.DecisionTreeRegressor(max_depth=4, random_state=0)
tree_sk.fit(X_train, y_train)                                               # обучение: перебор признаков и порогов
print(sklearn.tree.export_text(tree_sk, feature_names=["r1", "r2"], decimals=2))   # правила дерева текстом
mse_sk = sklearn.metrics.mean_squared_error(y_test, tree_sk.predict(X_test))
print(f"листьев: {tree_sk.get_n_leaves()}, MSE на тестовой выборке: {mse_sk:.0f}")

In [ ]:
# Результат на плоскости (r1, r2): точная поверхность и предсказание дерева scikit-learn
grid_points = np.column_stack([R1.ravel(), R2.ravel()])                      # узлы сетки: столбцы r1 и r2
E_tree = tree_sk.predict(grid_points).reshape(R1.shape)                      # предсказание дерева в узлах сетки
levels = np.arange(0, 631, 30)                                               # общая цветовая шкала, кДж/моль

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), sharey=True)
for ax, title, values in zip(axes, ["точная поверхность", f"дерево глубины {tree_sk.get_depth()}: MSE {mse_sk:.0f}"],
                             [energy(R1, R2), E_tree]):
    image = ax.contourf(R1, R2, values, levels=levels, cmap="RdYlBu_r", extend="both")
    ax.set_title(title)
    ax.set_xlabel("r$_1$, Å")
    ax.set_aspect("equal")
axes[0].set_ylabel("r$_2$, Å")
axes[1].scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="RdYlBu_r", vmin=0, vmax=630,
                s=12, edgecolors="black", linewidths=0.3)                    # обучающие точки в той же цветовой шкале
fig.colorbar(image, ax=axes, label="E, кДж/моль")
plt.show()

In [ ]:
# MSE на тестовой выборке в зависимости от глубины дерева и предсказание дерева выбранной глубины
show_depth = 8

depths = np.arange(1, 18)
models, mse_test = {}, {}
for depth in depths:
    models[depth] = sklearn.tree.DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_train, y_train)
    mse_test[depth] = sklearn.metrics.mean_squared_error(y_test, models[depth].predict(X_test))
    
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))

ax = axes[0]                                                              # слева: MSE от глубины
ax.plot(depths, list(mse_test.values()), "o-", color="#DC602E", lw=1.2, ms=4, label="тестовая выборка")
ax.plot(show_depth, mse_test[show_depth], "o", color="#3B6EA8", ms=8, label="выбранная глубина")
ax.axhline(noise**2, color="black", ls="--", lw=0.8, label="MSE шума")    # точнее шума предсказать нельзя
ax.set_yscale("log")
ax.set_xlabel("глубина дерева")
ax.set_ylabel("MSE, (кДж/моль)²")
ax.legend(frameon=False, loc="upper right")

ax = axes[1]                                                              # справа: предсказание выбранного дерева на плоскости
grid_points = np.column_stack([R1.ravel(), R2.ravel()])                   # узлы сетки: столбцы r1 и r2
E_show = models[show_depth].predict(grid_points).reshape(R1.shape)
image = ax.contourf(R1, R2, E_show, levels=np.arange(0, 631, 30), cmap="RdYlBu_r", extend="both")
fig.colorbar(image, ax=ax, label="E, кДж/моль")
ax.set_xlabel("r$_1$, Å")
ax.set_ylabel("r$_2$, Å")
ax.set_aspect("equal")
ax.set_title(f"глубина {show_depth}: MSE {mse_test[show_depth]:.0f}")
plt.tight_layout()
plt.show()

## 2.2 Беггинг

**Беггинг** (bagging) - ансамбль из одинаковых моделей, обученных на немного разных данных. Для каждого дерева из 400 объектов обучающей выборки случайно выбирают 400 **с возвращением**. Один и тот же объект может попасть в выборку несколько раз, а около 37 % объектов не попадают вовсе. Такую выборку называют бутстреп-выборкой. На каждой бутстреп-выборке обучают своё дерево без ограничения глубины, а ответы всех деревьев усредняют.

In [ ]:
# Бутстреп-выборки вручную: для каждого дерева из 400 объектов выбираем 400 с возвращением
rng = np.random.default_rng(1)
n = len(X_train)
samples = [rng.integers(0, n, size=n) for _ in range(100)]    # номера объектов в выборке каждого из 100 деревьев

fig, axes = plt.subplots(1, 3, figsize=(12.5, 5.0), sharey=True, layout="constrained")
for ax, k in zip(axes, [1, 2, 100]):                          # выборки с номерами 1, 2 и 100
    chosen = np.isin(np.arange(n), samples[k - 1])            # объект попал в выборку?
    ax.scatter(X_train[chosen, 0], X_train[chosen, 1], color="#3B6EA8", s=12, label="выбран")
    ax.scatter(X_train[~chosen, 0], X_train[~chosen, 1], s=14, facecolors="none", edgecolors="#999999",
               linewidths=0.7, label="не выбран")
    ax.set_title(f"выборка {k}: выбрано {chosen.sum()} из {n}")
    ax.set_xlabel("r$_1$, Å")
    ax.set_aspect("equal")
axes[0].set_ylabel("r$_2$, Å")
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncols=2, frameon=False)   # общая легенда под рисунком
plt.show()

In [ ]:
# Беггинг вручную
trees = [sklearn.tree.DecisionTreeRegressor(random_state=k).fit(X_train[sample], y_train[sample])
         for k, sample in enumerate(samples)]                 # своё дерево для каждой бутстреп-выборки
answers = np.array([tree.predict(X_test) for tree in trees])  # ответы каждого дерева на тестовой выборке
mse_trees = ((answers - y_test) ** 2).mean(axis=1)            # MSE каждого дерева
mse_mean = np.mean((answers.mean(axis=0) - y_test) ** 2)      # MSE среднего ответа всех деревьев
print(f"MSE одного дерева: от {mse_trees.min():.0f} до {mse_trees.max():.0f}, в среднем {mse_trees.mean():.0f}")
print(f"MSE среднего ответа {len(trees)} деревьев: {mse_mean:.0f}")

In [ ]:
# Результат на плоскости (r1, r2): деревья для выборок 1, 2 и 100 и среднее всех деревьев
grid_points = np.column_stack([R1.ravel(), R2.ravel()])                   # узлы сетки: столбцы r1 и r2
levels = np.arange(0, 631, 30)                                            # общая цветовая шкала, кДж/моль
panels = [(f"дерево {k}\nMSE {mse_trees[k - 1]:.0f}", trees[k - 1].predict(grid_points)) for k in [1, 2, 100]]
panels.append((f"среднее {len(trees)} деревьев\nMSE {mse_mean:.0f}",
               np.mean([tree.predict(grid_points) for tree in trees], axis=0)))   # средний ответ всех деревьев

fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.8), sharey=True)
for ax, (title, values) in zip(axes, panels):
    image = ax.contourf(R1, R2, values.reshape(R1.shape), levels=levels, cmap="RdYlBu_r", extend="both")
    ax.set_title(title)
    ax.set_xlabel("r$_1$, Å")
    ax.set_aspect("equal")
axes[0].set_ylabel("r$_2$, Å")
fig.colorbar(image, ax=axes, label="E, кДж/моль")
plt.show()

**Случайный лес** (Random Forest, RF) -  беггинг деревьев с ещё одним источником случайности. При каждом делении дерево ищет лучший порог не по всем признакам, а только по случайно выбранной их части. Сколько признаков просматривать, задаёт параметр `max_features`. Так деревья получаются ещё менее похожими друг на друга, что может улучшать беггинг. У нас признаков два, поэтому при каждом делении дерево берёт один из них наугад. В scikit-learn `RandomForestRegressor` по умолчанию просматривает все признаки (`max_features=1.0`), то есть без этого параметра работает как беггинг деревьев.

In [ ]:
# Случайный лес вручную: те же выборки, но при каждом делении дерево берёт один случайный признак
forest_trees = [sklearn.tree.DecisionTreeRegressor(max_features=1, random_state=k).fit(X_train[sample], y_train[sample])
                for k, sample in enumerate(samples)]          # max_features=1: порог ищут только по одному случайному признаку
forest_answers = np.array([tree.predict(X_test) for tree in forest_trees])
mse_forest_trees = ((forest_answers - y_test) ** 2).mean(axis=1)                 # MSE каждого дерева
mse_forest = np.mean((forest_answers.mean(axis=0) - y_test) ** 2)                # MSE среднего ответа всех деревьев
print(f"бэггинг: одно дерево в среднем {mse_trees.mean():.0f}, среднее 100 деревьев {mse_mean:.0f}")
print(f"случайный лес: одно дерево в среднем {mse_forest_trees.mean():.0f}, среднее 100 деревьев {mse_forest:.0f}")

In [ ]:
# Результат на плоскости (r1, r2): деревья случайного леса для выборок 1, 2 и 100 и среднее всех деревьев
grid_points = np.column_stack([R1.ravel(), R2.ravel()])                   # узлы сетки: столбцы r1 и r2
levels = np.arange(0, 631, 30)                                            # общая цветовая шкала, кДж/моль
panels = [(f"дерево {k}\nMSE {mse_forest_trees[k - 1]:.0f}", forest_trees[k - 1].predict(grid_points)) for k in [1, 2, 100]]
panels.append((f"среднее {len(forest_trees)} деревьев\nMSE {mse_forest:.0f}",
               np.mean([tree.predict(grid_points) for tree in forest_trees], axis=0)))   # средний ответ всех деревьев

fig, axes = plt.subplots(1, 4, figsize=(12.5, 3.8), sharey=True)
for ax, (title, values) in zip(axes, panels):
    image = ax.contourf(R1, R2, values.reshape(R1.shape), levels=levels, cmap="RdYlBu_r", extend="both")
    ax.set_title(title)
    ax.set_xlabel("r$_1$, Å")
    ax.set_aspect("equal")
axes[0].set_ylabel("r$_2$, Å")
fig.colorbar(image, ax=axes, label="E, кДж/моль")
plt.show()

In [ ]:
# Простой способ построить случайный лес - scikit-learn
for max_features in [2, 1]:                                               # сколько признаков просматривать при каждом делении
    forest = sklearn.ensemble.RandomForestRegressor(n_estimators=100, max_features=max_features, oob_score=True, random_state=0)
    forest.fit(X_train, y_train)
    mse_test = sklearn.metrics.mean_squared_error(y_test, forest.predict(X_test))
    mse_oob = np.mean((forest.oob_prediction_ - y_train) ** 2)            # оценка по объектам, не попавшим в выборку дерева
    print(f"max_features = {max_features}: MSE на тестовой выборке {mse_test:.0f}, оценка по невыбранным объектам {mse_oob:.0f}")

In [ ]:
# MSE на тестовой выборке в зависимости от числа деревьев и предсказание выбранного леса
show_trees = 100                                                          # число деревьев в лесе, который рисуем справа
show_features = 1                                                         # сколько признаков просматривать при каждом делении: 1 или 2

n_trees = np.arange(1, 201)
forests, curves = {}, {}
for max_features in [2, 1]:
    forests[max_features] = sklearn.ensemble.RandomForestRegressor(n_estimators=200, max_features=max_features,
                                                                   random_state=0).fit(X_train, y_train)
    answers_f = np.array([tree.predict(X_test) for tree in forests[max_features].estimators_])       # ответы каждого дерева
    curves[max_features] = ((np.cumsum(answers_f, axis=0) / n_trees[:, None] - y_test) ** 2).mean(axis=1)   # MSE первых K деревьев
mse_show = curves[show_features][show_trees - 1]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))

ax = axes[0]                                                              # слева: MSE от числа деревьев
ax.plot(n_trees, curves[2], color="#3B6EA8", lw=1.2, label="max_features = 2: все признаки (бэггинг)")
ax.plot(n_trees, curves[1], color="#DC602E", lw=1.2, label="max_features = 1: один случайный признак")
ax.plot(show_trees, mse_show, "o", color="black", ms=7, label="выбранный лес")
ax.axhline(noise**2, color="black", ls="--", lw=0.8, label="MSE шума")    # точнее шума предсказать нельзя
ax.set_xscale("log")
ax.set_ylim(0, 2000)
ax.set_xlabel("число деревьев")
ax.set_ylabel("MSE, (кДж/моль)²")
ax.legend(frameon=False, loc="upper right")

ax = axes[1]                                                              # справа: средний ответ первых show_trees деревьев
grid_points = np.column_stack([R1.ravel(), R2.ravel()])                   # узлы сетки: столбцы r1 и r2
E_show = np.mean([tree.predict(grid_points) for tree in forests[show_features].estimators_[:show_trees]], axis=0).reshape(R1.shape)
image = ax.contourf(R1, R2, E_show, levels=np.arange(0, 631, 30), cmap="RdYlBu_r", extend="both")
fig.colorbar(image, ax=ax, label="E, кДж/моль")
ax.set_xlabel("r$_1$, Å")
ax.set_ylabel("r$_2$, Å")
ax.set_aspect("equal")
ax.set_title(f"max_features = {show_features}, деревьев {show_trees}: MSE {mse_show:.0f}")
plt.tight_layout()
plt.show()

При двух признаках случайный выбор только мешает, поэтому лес с `max_features = 1` получается хуже беггинга. Выигрыш появляется, когда признаков много, как у дескрипторов в главе 3. Оценка по невыбранным объектам считается для каждого объекта по тем деревьям, в выборку которых он не попал, и поэтому не требует отдельной тестовой выборки.

## 2.3 Бустинг

**Бустинг** (boosting) - ансамбль, в котором модели обучают по очереди, и каждая следующая исправляет ошибки уже построенных. Первую модель обучают на самих данных. Затем на каждом шаге считают **остатки** $y - \hat{y}$, то есть разность между правильным ответом и текущим предсказанием, обучают на них следующую модель и добавляют её ответ к предсказанию.

In [ ]:
# Бустинг из многих деревьев: каждое следующее дерево учится на остатках всех предыдущих
n_trees = 200                                                # сколько деревьев построить
show_trees = [10, 100, 200]                                  # после скольких деревьев рисовать поверхность

grid_points = np.column_stack([R1.ravel(), R2.ravel()])       # узлы сетки: столбцы r1 и r2
pred_train = np.zeros(len(y_train))                           # сумма ответов уже построенных деревьев на обучающей выборке
pred_test = np.zeros(len(y_test))                             # то же на тестовой выборке
pred_grid = np.zeros(len(grid_points))                        # то же в узлах сетки
surfaces, mse_boost_train, mse_boost = {}, [], []
for k in range(1, n_trees + 1):
    residual = y_train - pred_train                           # остатки: данные минус предсказание (для первого дерева - сами данные)
    tree = sklearn.tree.DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_train, residual)   # дерево k учится на остатках
    pred_train += tree.predict(X_train)                       # добавляем ответ дерева к сумме
    pred_test += tree.predict(X_test)
    pred_grid += tree.predict(grid_points)
    mse_boost_train.append(np.mean((pred_train - y_train) ** 2))   # MSE после k деревьев
    mse_boost.append(np.mean((pred_test - y_test) ** 2))
    if k in show_trees:
        surfaces[k] = pred_grid.reshape(R1.shape).copy()      # запоминаем поверхность после k деревьев

In [ ]:
# Поверхности на плоскости (r1, r2): исходная и предсказания после show_trees деревьев
fig, axes = plt.subplots(1, len(show_trees) + 1, figsize=(12.5, 4.2), sharey=True)
panels = [("исходная поверхность", energy(R1, R2))]
panels += [(f"деревьев: {k}\nMSE обуч. {mse_boost_train[k - 1]:.0f}\nMSE тест. {mse_boost[k - 1]:.0f}", surfaces[k]) for k in show_trees]
for ax, (title, values) in zip(axes, panels):
    image = ax.contourf(R1, R2, values, levels=np.arange(0, 631, 30), cmap="RdYlBu_r", extend="both")   # общая цветовая шкала, кДж/моль
    ax.set_title(title)
    ax.set_xlabel("r$_1$, Å")
    ax.set_aspect("equal")
axes[0].set_ylabel("r$_2$, Å")
fig.colorbar(image, ax=axes, label="E, кДж/моль")
plt.show()

In [ ]:
# MSE в зависимости от числа деревьев
counts = np.arange(1, len(mse_boost) + 1)                                 # число деревьев: 1, 2, ..., n_trees
best = int(np.argmin(mse_boost)) + 1                                      # число деревьев с наименьшей MSE на тестовой выборке

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(counts, mse_boost_train, color="#3B6EA8", lw=1.2, label="обучающая выборка")
ax.plot(counts, mse_boost, color="#DC602E", lw=1.2, label="тестовая выборка")
ax.plot(best, mse_boost[best - 1], "o", color="black", ms=6, label=f"минимум: деревьев {best}, MSE {mse_boost[best - 1]:.0f}")
ax.axhline(noise**2, color="black", ls="--", lw=0.8, label="MSE шума")    # точнее шума предсказать нельзя
ax.set_xscale("log")
ax.set_ylim(0, 5200)
ax.set_xlabel("число деревьев")
ax.set_ylabel("MSE, (кДж/моль)²")
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

**Градиентный бустинг** (gradient boosting) - бустинг, в котором каждая следующая модель обучается на производной функции потерь по текущему предсказанию, взятой со знаком минус. Для MSE эта производная с точностью до множителя равна остатку $y - \hat{y}$, поэтому бустинг на остатках из предыдущих ячеек уже является градиентным бустингом. Обычно ответ следующей модели добавляют с множителем $\eta$ (скорость обучения), то есть за один шаг исправляют только часть ошибки, а начальным предсказанием служит среднее значение $y$ обучающей выборки. При $\eta = 1$ получается бустинг из предыдущих ячеек.

In [ ]:
# Градиентный бустинг: тот же цикл, но ответ каждого дерева добавляется с множителем η
n_trees = 200                                                # сколько деревьев построить
show_trees = [10, 100, 200]                                  # после скольких деревьев рисовать поверхность
learning_rate = 0.1                                           # скорость обучения η

grid_points = np.column_stack([R1.ravel(), R2.ravel()])       # узлы сетки: столбцы r1 и r2
pred_train = np.full(len(y_train), y_train.mean())            # начальное предсказание: средняя энергия обучающей выборки
pred_test = np.full(len(y_test), y_train.mean())              # то же на тестовой выборке
pred_grid = np.full(len(grid_points), y_train.mean())         # то же в узлах сетки
surfaces_grad, mse_grad_train, mse_grad = {}, [], []
for k in range(1, n_trees + 1):
    residual = y_train - pred_train                           # остатки: производная MSE по предсказанию со знаком минус (без множителя 2/n)
    tree = sklearn.tree.DecisionTreeRegressor(max_depth=2, random_state=0).fit(X_train, residual)   # дерево k учится на остатках
    pred_train += learning_rate * tree.predict(X_train)       # шаг градиентного спуска: ответ дерева с множителем η
    pred_test += learning_rate * tree.predict(X_test)
    pred_grid += learning_rate * tree.predict(grid_points)
    mse_grad_train.append(np.mean((pred_train - y_train) ** 2))    # MSE после k деревьев
    mse_grad.append(np.mean((pred_test - y_test) ** 2))
    if k in show_trees:
        surfaces_grad[k] = pred_grid.reshape(R1.shape).copy()      # запоминаем поверхность после k деревьев

# Поверхности на плоскости (r1, r2): исходная и предсказания после show_trees деревьев
fig, axes = plt.subplots(1, len(show_trees) + 1, figsize=(12.5, 4.2), sharey=True)
panels = [("исходная поверхность", energy(R1, R2))]
panels += [(f"деревьев: {k}\nMSE обуч. {mse_grad_train[k - 1]:.0f}\nMSE тест. {mse_grad[k - 1]:.0f}", surfaces_grad[k]) for k in show_trees]
for ax, (title, values) in zip(axes, panels):
    image = ax.contourf(R1, R2, values, levels=np.arange(0, 631, 30), cmap="RdYlBu_r", extend="both")   # общая цветовая шкала, кДж/моль
    ax.set_title(title)
    ax.set_xlabel("r$_1$, Å")
    ax.set_aspect("equal")
axes[0].set_ylabel("r$_2$, Å")
fig.colorbar(image, ax=axes, label="E, кДж/моль")
plt.show()

In [ ]:
# MSE в зависимости от числа деревьев: бустинг и градиентный бустинг
fig, ax = plt.subplots(figsize=(11, 3.6))
for name, color, mse_train, mse_test in [("бустинг", "#DC602E", mse_boost_train, mse_boost),
                                         ("градиентный бустинг", "#3B6EA8", mse_grad_train, mse_grad)]:
    counts = np.arange(1, len(mse_test) + 1)                              # число деревьев: 1, 2, ..., n_trees
    best = int(np.argmin(mse_test)) + 1                                   # число деревьев с наименьшей MSE на тестовой выборке
    ax.plot(counts, mse_train, color=color, lw=1.2, ls=":", label=f"{name}: обучающая выборка")
    ax.plot(counts, mse_test, color=color, lw=1.2, label=f"{name}: тестовая выборка")
    ax.plot(best, mse_test[best - 1], "o", color=color, ms=6)             # точка: минимум на тестовой выборке
    print(f"{name}: минимум MSE на тестовой выборке {mse_test[best - 1]:.0f} при {best} деревьях")
ax.axhline(noise**2, color="black", ls="--", lw=0.8, label="MSE шума")    # точнее шума предсказать нельзя
ax.set_xscale("log")
ax.set_ylim(0, 5200)
ax.set_xlabel("число деревьев")
ax.set_ylabel("MSE, (кДж/моль)²")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Простой способ построить бустинг - scikit-learn
for learning_rate in [1.0, 0.1]:                                          # скорость обучения η=1.0 - обычный бустинг
    boosting = sklearn.ensemble.GradientBoostingRegressor(n_estimators=200, learning_rate=learning_rate, max_depth=2, random_state=0)
    boosting.fit(X_train, y_train)                                        # обучение: 200 деревьев глубины 2 по очереди на остатках
    mse_train = sklearn.metrics.mean_squared_error(y_train, boosting.predict(X_train))
    mse_test = sklearn.metrics.mean_squared_error(y_test, boosting.predict(X_test))
    print(f"learning_rate = {learning_rate}: MSE на обучающей выборке {mse_train:.0f}, на тестовой {mse_test:.0f}")

## 2.4 Стекинг

**Стекинг** (stacking) - ансамбль из разных моделей, ответы которых объединяет ещё одна модель. Сначала на обучающей выборке независимо обучают несколько **базовых моделей**. Из их ответов формируют новую матрицу признаков, на которой обучают **метамодель**, часто обычную линейную регрессию: $\hat{y} = w_1 \hat{y}_1 + w_2 \hat{y}_2 + b$. Веса $w$ показывают, какой базовой модели метамодель доверяет больше.

In [ ]:
# Стекинг: две базовые модели
forest = sklearn.ensemble.RandomForestRegressor(n_estimators=100, random_state=0).fit(X_train, y_train)   # базовая модель 1: случайный лес
boosting = sklearn.ensemble.GradientBoostingRegressor(n_estimators=200, 
                                                      learning_rate=0.1, 
                                                      max_depth=2,
                                                      random_state=0).fit(X_train, y_train)               # базовая модель 2: градиентный бустинг

In [ ]:
# Ответы базовых моделей - новые признаки для метамодели
answers_train = np.column_stack([forest.predict(X_train), boosting.predict(X_train)])   # два столбца: ответ леса и ответ бустинга
answers_test = np.column_stack([forest.predict(X_test), boosting.predict(X_test)])
pd.DataFrame({"лес": answers_train[:, 0], 
              "бустинг": answers_train[:, 1], 
              "E": y_train}).head().round(1)   # новые признаки и правильный ответ

In [ ]:
# Метамодель на torch: один нейрон без функции активации, E = w1·лес + w2·бустинг + b
torch.manual_seed(0)                                                         # начальные веса воспроизводимы
meta = torch.nn.Sequential(torch.nn.Linear(2, 1))                            # 2 признака (ответы леса и бустинга), 1 выход
A_t = torch.tensor(answers_train, dtype=torch.float32)                       # новые признаки: тензор 400 × 2
y_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)                # правильные ответы: тензор 400 × 1

In [ ]:
ml_tool.show(meta)

In [ ]:
# Обучение метамодели по всей обучающей выборке методом L-BFGS
optimizer = torch.optim.LBFGS(meta.parameters(), max_iter=1000)              # настройки обучения: не больше 1000 итераций

def closure():                                                               # MSE и её производные по весам и смещению
    optimizer.zero_grad()
    mse = torch.nn.functional.mse_loss(meta(A_t), y_t)
    mse.backward()
    return mse

optimizer.step(closure)                                                      # подбор весов и смещения
w1, w2 = meta[0].weight[0].tolist()                                          # веса нейрона
b = meta[0].bias.item()                                                      # смещение нейрона
print(f"метамодель: E = {w1:.2f}·лес + {w2:.2f}·бустинг {b:+.1f}")

In [ ]:
# MSE базовых моделей и стекинга
def predict_meta(answers):                                                   # предсказание метамодели для массива NumPy
    with torch.no_grad():
        return meta(torch.tensor(answers, dtype=torch.float32)).numpy()[:, 0]

for name, pred_train, pred_test in [("случайный лес", answers_train[:, 0], answers_test[:, 0]),
                                    ("градиентный бустинг", answers_train[:, 1], answers_test[:, 1]),
                                    ("стекинг", predict_meta(answers_train), predict_meta(answers_test))]:
    print(f"{name:20s} MSE на обучающей выборке {np.mean((pred_train - y_train) ** 2):4.0f}, на тестовой {np.mean((pred_test - y_test) ** 2):4.0f}")

Стекинг на тестовой выборке оказался хуже лучшей из базовых моделей, бустинга. Метамодель почти полностью полагается на предсказание случайного леса, который хорошо запомнил обучающую выборку. Чтобы метамодель оценивала базовые модели честно, ей дают ответы для объектов, которых базовые модели при обучении не видели. Для этого обучающую выборку делят на пять подвыборок, модели обучают на четырёх подвыборках и записывают их ответы для пятой, и так по очереди для каждой подвыборки.

Ответы, полученные от пяти моделей случайного леса и пяти моделей бустинга, используют для обучения метамодели. После этого для тестовой выборки сначала считают предсказания случайного леса и бустинга, обученных на всей обучающей выборке (как на предыдущем этапе), а затем эти предсказания передают в метамодель (обученную иначе, чем на предыдущем этапе).

In [ ]:
# Разделение обучающей выборки на 5 подвыборок
X_parts = np.array_split(X_train, 5)                          # список из 5 массивов признаков: объекты 0-79, 80-159, ...
y_parts = np.array_split(y_train, 5)                          # список из 5 массивов энергий тех же объектов

print(f"Обучающая выборка: {len(X_train)}")
for k in range(5):
    print(f"Подвыборка {k + 1}: {len(X_parts[k])}")

In [ ]:
# Ответы базовых моделей для объектов, которых они не видели: обучение на четырёх подвыборках, ответы для пятой
answers_parts = []                                            # ответы двух моделей для каждой подвыборки
for k in range(5):
    X_seen = np.concatenate([X_parts[j] for j in range(5) if j != k])   # четыре подвыборки для обучения
    y_seen = np.concatenate([y_parts[j] for j in range(5) if j != k])
    forest_part = sklearn.base.clone(forest).fit(X_seen, y_seen)        # clone: такая же модель, но необученная
    boosting_part = sklearn.base.clone(boosting).fit(X_seen, y_seen)
    answers = np.column_stack([forest_part.predict(X_parts[k]), boosting_part.predict(X_parts[k])])   # ответы для отложенной подвыборки
    answers_parts.append(answers)
    mse_parts = np.mean((answers - y_parts[k][:, None]) ** 2, axis=0)   # MSE двух моделей на отложенной подвыборке
    print(f"Подвыборка {k + 1} отложена: обучение на {len(X_seen)} объектах, MSE леса {mse_parts[0]:.0f}, бустинга {mse_parts[1]:.0f}")

answers_train = np.concatenate(answers_parts)                 # новые признаки для всех 400 объектов: столбцы «лес» и «бустинг»
mse_all = np.mean((answers_train - y_train[:, None]) ** 2, axis=0)
print(f"Все объекты: MSE леса {mse_all[0]:.0f}, бустинга {mse_all[1]:.0f}")

In [ ]:
# Метамодель на ответах для невиденных объектов: тот же нейрон и то же обучение, что выше
torch.manual_seed(0)                                                         # начальные веса воспроизводимы
meta = torch.nn.Sequential(torch.nn.Linear(2, 1))                            # новая необученная метамодель
A_t = torch.tensor(answers_train, dtype=torch.float32)                       # новые признаки для невиденных объектов: тензор 400 × 2

optimizer = torch.optim.LBFGS(meta.parameters(), max_iter=1000)              # настройки обучения: не больше 1000 итераций
optimizer.step(closure)                                                      # closure из ячейки выше: MSE и её производные по весам и смещению
w1, w2 = meta[0].weight[0].tolist()                                          # веса нейрона
b = meta[0].bias.item()                                                      # смещение нейрона
print(f"метамодель: E = {w1:.2f}·лес + {w2:.2f}·бустинг {b:+.1f}")

In [ ]:
# MSE базовых моделей и стекинга: на отложенных подвыборках и на тестовой выборке
for name, pred_parts, pred_test in [("случайный лес", answers_train[:, 0], answers_test[:, 0]),
                                    ("градиентный бустинг", answers_train[:, 1], answers_test[:, 1]),
                                    ("стекинг", predict_meta(answers_train), predict_meta(answers_test))]:   # predict_meta из ячейки выше
    print(f"{name:20s} MSE на отложенных подвыборках {np.mean((pred_parts - y_train) ** 2):4.0f}, на тестовой выборке {np.mean((pred_test - y_test) ** 2):4.0f}")
mse_stack = np.mean((predict_meta(answers_test) - y_test) ** 2)              # MSE стекинга на тестовой выборке

## 2.5 Время обучения и использования моделей

| Модель | Обучение | Предсказание 2000 объектов | Размер модели | MSE |
|---|---|---|---|---|
| **Занятие 10** | | | | |
| линейная регрессия | 0.4 мс | 0.08 мс | 3 параметра | 4823 |
| нейронная сеть 2-8-1, 1000 эпох | 7.8 с | 0.09 мс | 33 параметра | 333 |
| нейронная сеть 2-8-8-1, 1500 эпох | 13.6 с | 0.12 мс | 105 параметров | 303 |
| **Занятие 11** | | | | |
| дерево решений без ограничения глубины | 0.8 мс | 0.15 мс | 799 узлов | 891 |
| случайный лес, 100 деревьев | 69 мс | 14 мс | 50 340 узлов | 474 |
| градиентный бустинг, 200 деревьев глубины 2 | 62 мс | 2.6 мс | 1386 узлов | 400 |
| стекинг (случайный лес и градиентный бустинг) | 790 мс | 17 мс | обе модели | 373 |

# 🎯 Глава 3. Предсказание растворимости соединений

Попытаемся воспроизвести результат работы из оригинальной статьи ([Sorkun et al., 2021](https://doi.org/10.1016/j.isci.2020.101961)), в которой растворимость соединений в воде log S (S в моль/л) предсказывают по их дескрипторам. 

Для обучения, валидации и тестирования будут использованы два набора:
- **Набор AqSolDB** - 9982 соединения с экспериментальной растворимостью в воде, объединённые из девяти открытых источников. Кроме органических соединений, в нём есть соли и неорганические вещества, а разброс экспериментальных значений составляет около 0.5 log S. Из него берутся обучающая и валидационная выборки.
- **Набор Huuskonen** - 1291 органическое соединение, самая надёжная часть AqSolDB с разбросом экспериментальных значений 0.27 log S. Он служит тестовой выборкой в оригинальной работе и в нашем эксперименте. 

In [ ]:
# Набор данных AqSolDB (Sorkun et al., 2019, CC0): https://doi.org/10.7910/DVN/OVHAW8
# !curl -sSL -o AqSolDB.csv "https://dataverse.harvard.edu/api/access/datafile/3407241?format=original"
# Набор Huuskonen - часть AqSolDB, тестовая выборка статьи (CC0): https://github.com/mcsorkun/AqSolDB
# !curl -sSL -o dataset-E.csv "https://raw.githubusercontent.com/mcsorkun/AqSolDB/master/data/dataset-E.csv"

### 3.1 Подготовка данных

In [ ]:
dataset = pd.read_csv("AqSolDB.csv")              # весь набор AqSolDB
huuskonen = pd.read_csv("dataset-E.csv")          # набор Huuskonen: тестовая выборка, как в статье
print(f"AqSolDB: {len(dataset)} соединений, набор Huuskonen: {len(huuskonen)}")
dataset.head()

In [ ]:
# Берём только ID, InChIKey, Name, SMILES и log S
columns = ["ID", "InChIKey", "Name", "SMILES", "Solubility"]
dataset = dataset[columns]
huuskonen = huuskonen[columns]
dataset

In [ ]:
# Обучающий набор как в статье: без соединений набора Huuskonen и без двух источников с наибольшим разбросом значений (A и F)
in_huuskonen = dataset["InChIKey"].str[:14].isin(huuskonen["InChIKey"].str[:14])          # соединение есть в наборе Huuskonen: совпадает InChIKey
noisy_source = dataset["ID"].str[0].isin(["A", "F"])                         # источник данных - первая буква ID
print(f"AqSolDB: {len(dataset)}, из них совпадают по строению с набором Huuskonen: {in_huuskonen.sum()}, из остальных в источниках A и F: {(noisy_source & ~in_huuskonen).sum()}")

train_part = dataset[~in_huuskonen & ~noisy_source].copy()
train_part["выборка"] = "обучение"
huuskonen["выборка"] = "тест"
dataset = pd.concat([train_part, huuskonen], ignore_index=True)              # одна таблица: соединения для обучения и тестовая выборка
dataset["выборка"].value_counts()

In [ ]:
# Удалить строки с невалидными SMILES
RDLogger.DisableLog("rdApp.*") # не печатать предупреждения RDKit для каждой строки
n_before = len(dataset)
dataset = dataset[dataset["SMILES"].apply(lambda s: Chem.MolFromSmiles(s) is not None)].reset_index(drop=True)
print(f"удалено строк: {n_before - len(dataset)}, осталось: {len(dataset)}")
dataset.head()

In [ ]:
# Распределение данных
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.hist(dataset["Solubility"], bins=60, color="#3B6EA8", lw=0)
ax.axvline(dataset["Solubility"].median(), 
           color="black", ls="--", lw=0.8,
           label=f"медиана {dataset['Solubility'].median():.2f}")
ax.set_xlabel("log S")
ax.set_ylabel("число соединений")
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

# Три наименее и три наиболее растворимых соединения
extremes = dataset.sort_values("Solubility").iloc[[0, 1, 2, -3, -2, -1]]
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in extremes["SMILES"]], molsPerRow=3, subImgSize=(250, 200),
                     legends=[f"log S = {s:.2f}" for s in extremes["Solubility"]])

### 3.2 Расчёт дескрипторов 

In [ ]:
# Расчёт дескрипторов RDKit для каждого SMILES
feature_names = ["MolWt",                # молярная масса, г/моль
                 "MolLogP",              # логарифм коэффициента распределения октанол/вода, log P (Wildman, Crippen)
                 "MolMR",                # молярная рефракция
                 "HeavyAtomCount",       # число тяжёлых атомов (все, кроме H)
                 "NumHAcceptors",        # число акцепторов водородной связи
                 "NumHDonors",           # число доноров водородной связи
                 "NumHeteroatoms",       # число атомов, кроме C и H
                 "NumRotatableBonds",    # число вращающихся связей
                 "NumValenceElectrons",  # число валентных электронов
                 "NumAromaticRings",     # число ароматических циклов
                 "NumSaturatedRings",    # число насыщенных циклов
                 "NumAliphaticRings",    # число неароматических циклов
                 "RingCount",            # общее число циклов
                 "TPSA",                 # топологическая площадь полярной поверхности, Å²
                 "LabuteASA",            # площадь поверхности молекулы по вкладам атомов, Å²
                 "BalabanJ",             # топологический индекс Балабана
                 "BertzCT"               # индекс сложности Берца
                 ]

mols = [Chem.MolFromSmiles(s) for s in dataset["SMILES"]]
dataset[feature_names] = [[getattr(Descriptors, name)(mol) for name in feature_names] for mol in mols]
dataset[["SMILES", "Solubility"] + feature_names].head()

In [ ]:
# Зависимость растворимости от каждого дескриптора; r - коэффициент корреляции Пирсона
r = dataset[feature_names].corrwith(dataset["Solubility"]).sort_values(key=abs, ascending=False)

fig, axes = plt.subplots(nrows=5, ncols=4, sharey=True, figsize=(12.5, 11))
for ax, name in zip(axes.flat, r.index):                       # панели по убыванию |r|
    ax.scatter(dataset[name], dataset["Solubility"], s=3, alpha=0.3, color="#3B6EA8", lw=0)
    ax.set_xlim(*dataset[name].quantile([0.001, 0.999]))      # без единичных очень больших молекул
    ax.set_xlabel(name)
    ax.set_title(f"r = {r[name]:.2f}")
for ax in axes[:, 0]:
    ax.set_ylabel("log S")
for ax in axes.flat[len(feature_names):]:                      # лишние пустые панели
    ax.remove()
plt.tight_layout()
plt.show()

In [ ]:
# Тестовая выборка - набор Huuskonen; остальные соединения делим на обучающую и валидационную выборки (80/20)
train_part = dataset[dataset["выборка"] == "обучение"]
test_part = dataset[dataset["выборка"] == "тест"]

X_train, X_val, y_train, y_val = sklearn.model_selection.train_test_split(train_part[feature_names], train_part["Solubility"].to_numpy(),
                                                                          test_size=0.2, random_state=0)
X_test, y_test = test_part[feature_names], test_part["Solubility"].to_numpy()
print(f"обучающая: {len(y_train)}, валидационная: {len(y_val)}, тестовая: {len(y_test)}")

In [ ]:
# Стандартизация признаков
scaler = sklearn.preprocessing.StandardScaler().fit(X_train)
X_train_std = scaler.transform(X_train)
X_val_std = scaler.transform(X_val)
X_test_std = scaler.transform(X_test)

### 3.3 Линейная модель

In [ ]:
# Линейная модель на torch: один нейрон без функции активации, log S = w1·x1 + ... + w17·x17 + b
torch.manual_seed(0)                                                         # начальные веса воспроизводимы
model_linear = torch.nn.Sequential(torch.nn.Linear(len(feature_names), 1))   # 17 признаков (дескрипторы), 1 выход
X_t = torch.tensor(X_train_std, dtype=torch.float32)                         # стандартизованные дескрипторы обучающей выборки
y_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)                # log S обучающей выборки

In [ ]:
ml_tool.show(model_linear, features=feature_names, output="log S", max_neurons=17)

In [ ]:
# Обучение линейной модели по всей обучающей выборке
optimizer = torch.optim.LBFGS(model_linear.parameters(), max_iter=1000)      # настройки обучения: не больше 1000 итераций

def closure():                                                               # MSE и её производные по весам и смещению
    optimizer.zero_grad()
    mse = torch.nn.functional.mse_loss(model_linear(X_t), y_t)
    mse.backward()
    return mse

optimizer.step(closure)                                                      # подбор весов и смещения
print(f"MSE на обучающей выборке: {closure().item():.3f}")

In [ ]:
# Метрики качества на тестовой выборке
def metrics(y_true, y_pred):   # MAE, RMSE и R²; MAE и RMSE в единицах log S
    return {"MAE": sklearn.metrics.mean_absolute_error(y_true, y_pred),
            "RMSE": sklearn.metrics.root_mean_squared_error(y_true, y_pred),
            "R²": sklearn.metrics.r2_score(y_true, y_pred)}

def predict(model, X):                                                       # предсказание модели torch для массива NumPy
    with torch.no_grad():
        return model(torch.tensor(X, dtype=torch.float32)).numpy()[:, 0]

mean_logS = dataset.loc[X_train.index, "Solubility"].mean()                  # среднее log S соединений обучающей выборки
val_test = {"среднее значение": np.full(len(y_test), mean_logS),              # точка отсчёта: один и тот же ответ для всех соединений
            "линейная модель": predict(model_linear, X_test_std)}
results = pd.DataFrame({name: metrics(y_test, pred) for name, pred in val_test.items()}).T   # строка таблицы для каждой модели
results.round(3)

### 3.4 Модель прямого распространения

In [ ]:
# Нейронная сеть 17-64-64-1: два скрытых слоя по 64 нейрона
torch.manual_seed(0)                       # начальные веса и порядок батчей воспроизводимы
model_net = torch.nn.Sequential(
    torch.nn.Linear(17, 64),               # первый скрытый слой: 17 признаков, 64 нейрона
    torch.nn.ReLU(),                       # функция активации первого скрытого слоя: ReLU
    torch.nn.Linear(64, 64),               # второй скрытый слой: 64 нейрона
    torch.nn.ReLU(),                       # функция активации второго скрытого слоя: ReLU
    torch.nn.Linear(64, 1))                # выходной слой: предсказание log S

In [ ]:
ml_tool.show(model_net, features=feature_names, output="log S")

In [ ]:
# Валидационная выборка в виде тензоров и разбиение обучающей выборки на батчи
X_v = torch.tensor(X_val_std, dtype=torch.float32)                           # дескрипторы валидационной выборки
y_v = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)                  # log S валидационной выборки
batches = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_t, y_t),
                                      batch_size=100, shuffle=True)          # батчи по 100 соединений, порядок соединений случайный

In [ ]:
# Обучение методом обратного распространения ошибки; модель получает веса лучшей эпохи по валидационной выборке
def train(net, n_epochs=200):
    optimizer = torch.optim.Adam(net.parameters(), lr=0.001)                 # оптимизатор Adam и скорость обучения
    curve_train, curve_val = [], []                                          # RMSE на двух выборках после каждой эпохи
    for epoch in range(n_epochs):
        for X_batch, y_batch in batches:                                     # один шаг обучения на каждый батч
            optimizer.zero_grad()                                            # обнуление производных предыдущего шага
            mse = torch.nn.functional.mse_loss(net(X_batch), y_batch)        # прямой проход и MSE на батче
            mse.backward()                                                   # обратный проход: производные MSE по весам и смещениям
            optimizer.step()                                                 # обновление весов и смещений
        with torch.no_grad():
            curve_train.append(torch.nn.functional.mse_loss(net(X_t), y_t).item() ** 0.5)   # RMSE на обучающей выборке
            curve_val.append(torch.nn.functional.mse_loss(net(X_v), y_v).item() ** 0.5)     # RMSE на валидационной выборке
        if curve_val[-1] == min(curve_val):
            best_weights = copy.deepcopy(net.state_dict())                   # запоминаем веса эпохи с наименьшей RMSE на валидационной выборке
    net.load_state_dict(best_weights)                                        # возвращаем модели веса лучшей эпохи
    return curve_train, curve_val

curve_train, curve_val = train(model_net)
best_epoch = int(np.argmin(curve_val)) + 1
print(f"лучшая эпоха: {best_epoch}, RMSE на обучающей выборке {curve_train[best_epoch - 1]:.3f}, на валидационной {curve_val[best_epoch - 1]:.3f}")

In [ ]:
# RMSE в зависимости от числа эпох
epochs = np.arange(1, len(curve_val) + 1)

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(epochs, curve_train, color="#3B6EA8", lw=1.2, label="обучающая выборка")
ax.plot(epochs, curve_val, color="#DC602E", lw=1.2, label="валидационная выборка")
ax.plot(best_epoch, curve_val[best_epoch - 1], "o", color="black", ms=6, label=f"лучшая эпоха: {best_epoch}, RMSE {curve_val[best_epoch - 1]:.2f}")
ax.set_ylim(0.6, 2.0)
ax.set_xlabel("эпоха")
ax.set_ylabel("RMSE, log S")
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Модель меньше: 17-16-1, один скрытый слой из 16 нейронов
torch.manual_seed(0)                       # начальные веса и порядок батчей воспроизводимы
model_net_small = torch.nn.Sequential(
    torch.nn.Linear(17, 16),               # скрытый слой: 17 признаков, 16 нейронов
    torch.nn.ReLU(),                       # функция активации скрытого слоя: ReLU
    torch.nn.Linear(16, 1))                # выходной слой: предсказание log S

curve_train_small, curve_val_small = train(model_net_small)
best_epoch_small = int(np.argmin(curve_val_small)) + 1
print(f"лучшая эпоха: {best_epoch_small}, RMSE на обучающей выборке {curve_train_small[best_epoch_small - 1]:.3f}, на валидационной {curve_val_small[best_epoch_small - 1]:.3f}")

In [ ]:
# Модель больше: 17-128-128-1, два скрытых слоя по 128 нейронов
torch.manual_seed(0)                       # начальные веса и порядок батчей воспроизводимы
model_net_large = torch.nn.Sequential(
    torch.nn.Linear(17, 128),              # первый скрытый слой: 17 признаков, 128 нейронов
    torch.nn.ReLU(),                       # функция активации первого скрытого слоя: ReLU
    torch.nn.Linear(128, 128),             # второй скрытый слой: 128 нейронов
    torch.nn.ReLU(),                       # функция активации второго скрытого слоя: ReLU
    torch.nn.Linear(128, 1))               # выходной слой: предсказание log S

curve_train_large, curve_val_large = train(model_net_large)
best_epoch_large = int(np.argmin(curve_val_large)) + 1
print(f"лучшая эпоха: {best_epoch_large}, RMSE на обучающей выборке {curve_train_large[best_epoch_large - 1]:.3f}, на валидационной {curve_val_large[best_epoch_large - 1]:.3f}")

In [ ]:
# Выбор лучшей нейронной сети по валидационной выборке
nets = {"17-16-1": model_net_small, "17-64-64-1": model_net, "17-128-128-1": model_net_large}                  # три обученные модели
results_val = pd.DataFrame({name: metrics(y_val, predict(net, X_val_std)) for name, net in nets.items()}).T   # метрики на валидационной выборке
best_name = results_val["RMSE"].idxmin()                                     # модель с наименьшей RMSE
print(f"лучшая нейронная сеть: {best_name}")
results_val.round(3)

In [ ]:
# Лучшая нейронная сеть на тестовой выборке и общий список
val_test[f"нейронная сеть прямого распространения"] = predict(nets[best_name], X_test_std)   # тестовая выборка используется только для выбранной модели
results = pd.DataFrame({name: metrics(y_test, pred) for name, pred in val_test.items()}).T   # строка таблицы для каждой модели
results.round(3)

### 3.5 Случайный лес

In [ ]:
# Случайный лес
model_forest = sklearn.ensemble.RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1)   # 300 деревьев, обучение на всех ядрах процессора
model_forest.fit(X_train_std, y_train)                                       # обучение: каждое дерево на своей бутстреп-выборке
print({name: round(value, 3) for name, value in metrics(y_val, model_forest.predict(X_val_std)).items()})   # метрики на валидационной выборке

In [ ]:
# Сколько признаков просматривать при каждом делении: леса с разным max_features
forests = {}
for max_features in [1, 2, 3, 4, 6, 9, 13, 17]:                              # 17 - все признаки, то есть беггинг
    forests[max_features] = sklearn.ensemble.RandomForestRegressor(n_estimators=300, max_features=max_features,
                                                                   random_state=0, n_jobs=-1).fit(X_train_std, y_train)

results_forest = pd.DataFrame({max_features: metrics(y_val, forest.predict(X_val_std))
                               for max_features, forest in forests.items()}).T   # метрики на валидационной выборке
results_forest.index.name = "max_features"
best_features = results_forest["RMSE"].idxmin()                              # значение с наименьшей RMSE
print(f"лучший случайный лес: max_features = {best_features}")
results_forest.round(3)

In [ ]:
# RMSE на валидационной выборке в зависимости от max_features и от числа деревьев
n_trees = np.arange(1, 301)
curves_forest = {}
for max_features in [best_features, 17]:                                     # лучший лес и лес со всеми признаками
    answers = np.array([tree.predict(X_val_std) for tree in forests[max_features].estimators_])            # ответы каждого дерева
    curves_forest[max_features] = np.sqrt(((np.cumsum(answers, axis=0) / n_trees[:, None] - y_val) ** 2).mean(axis=1))   # RMSE первых K деревьев

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))

ax = axes[0]                                                                 # слева: RMSE от max_features
ax.plot(results_forest.index, results_forest["RMSE"], "o-", color="#3B6EA8", lw=1.2, ms=4)
ax.plot(best_features, results_forest.loc[best_features, "RMSE"], "o", color="black", ms=7, label=f"лучшее значение: {best_features}")
ax.set_xlabel("max_features")
ax.set_ylabel("RMSE на валидационной выборке, log S")
ax.legend(frameon=False, loc="upper center")

ax = axes[1]                                                                 # справа: RMSE от числа деревьев
ax.plot(n_trees, curves_forest[17], color="#3B6EA8", lw=1.2, label="max_features = 17: все признаки (беггинг)")
ax.plot(n_trees, curves_forest[best_features], color="#DC602E", lw=1.2, label=f"max_features = {best_features}")
ax.set_xscale("log")
ax.set_ylim(0.8, 1.4)
ax.set_xlabel("число деревьев")
ax.set_ylabel("RMSE на валидационной выборке, log S")
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Лучший случайный лес на тестовой выборке и общий список
val_test["случайный лес"] = forests[best_features].predict(X_test_std)                    # тестовая выборка используется только для выбранной модели
results = pd.DataFrame({name: metrics(y_test, pred) for name, pred in val_test.items()}).T   # строка таблицы для каждой модели
results.round(3)

### 3.6 Градиентный бустинг



In [ ]:
# Градиентный бустинг
model_boosting = sklearn.ensemble.GradientBoostingRegressor(n_estimators=500, learning_rate=0.1, max_depth=3, random_state=0)   # 500 деревьев глубины 3, скорость обучения 0.1
model_boosting.fit(X_train_std, y_train)                                     # обучение: деревья строятся по очереди на остатках
print({name: round(value, 3) for name, value in metrics(y_val, model_boosting.predict(X_val_std)).items()})   # метрики на валидационной выборке

In [ ]:
# Скорость обучения и число деревьев: RMSE на валидационной выборке после каждого дерева
boostings = {0.1: model_boosting}                                            # уже обученная модель
for learning_rate in [1.0, 0.3]:
    boostings[learning_rate] = sklearn.ensemble.GradientBoostingRegressor(n_estimators=500, 
                                                                          learning_rate=learning_rate, 
                                                                          max_depth=3,
                                                                          random_state=0).fit(X_train_std, y_train)

curves_boosting = {learning_rate: np.array([sklearn.metrics.root_mean_squared_error(y_val, answer) for answer in boosting.staged_predict(X_val_std)])
                   for learning_rate, boosting in boostings.items()}          # RMSE после 1, 2, ... деревьев
results_boosting = pd.DataFrame({learning_rate: {"лучшее число деревьев": int(np.argmin(curve)) + 1, "RMSE": curve.min()}
                                 for learning_rate, curve in curves_boosting.items()}).T.sort_index(ascending=False)
results_boosting["лучшее число деревьев"] = results_boosting["лучшее число деревьев"].astype(int)
results_boosting.index.name = "learning_rate"
best_rate = results_boosting["RMSE"].idxmin()                                # скорость обучения с наименьшей RMSE
best_trees = results_boosting.loc[best_rate, "лучшее число деревьев"]        # и число деревьев для неё
print(f"лучший градиентный бустинг: learning_rate = {best_rate}, деревьев {best_trees}")
results_boosting.round(3)

In [ ]:
# RMSE на валидационной выборке в зависимости от числа деревьев
n_trees = np.arange(1, 501)

fig, ax = plt.subplots(figsize=(8, 3.6))
for learning_rate, color in zip([1.0, 0.3, 0.1], ["#DC602E", "#6B8E23", "#3B6EA8"]):
    ax.plot(n_trees, curves_boosting[learning_rate], color=color, lw=1.2, label=f"η = {learning_rate}")
ax.plot(best_trees, curves_boosting[best_rate][best_trees - 1], "o", color="black", ms=6, label=f"лучшая модель: η = {best_rate}, деревьев {best_trees}")
ax.set_xscale("log")
ax.set_ylim(0.85, 2.0)
ax.set_xlabel("число деревьев")
ax.set_ylabel("RMSE на валидационной выборке, log S")
ax.legend(frameon=False, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Лучший градиентный бустинг на тестовой выборке и общий список
val_test["градиентный бустинг"] = list(boostings[best_rate].staged_predict(X_test_std))[best_trees - 1]   # предсказание первых best_trees деревьев
results = pd.DataFrame({name: metrics(y_test, pred) for name, pred in val_test.items()}).T   # строка таблицы для каждой модели
results.round(3)

### 3.7 Консенсус и стекинг

In [ ]:
# Cреднее предсказаний трёх лучших моделей
val_test["консенсус трёх моделей"] = (predict(nets[best_name], X_test_std)
                                      + forests[best_features].predict(X_test_std)
                                      + list(boostings[best_rate].staged_predict(X_test_std))[best_trees - 1]) / 3
results = pd.DataFrame({name: metrics(y_test, pred) for name, pred in val_test.items()}).T   # строка таблицы для каждой модели
results.round(3)

In [ ]:
# Стекинг: ответы трёх лучших моделей - новые признаки для метамодели
def answers(X):                                                              # ответы нейронной сети, случайного леса и градиентного бустинга
    return np.column_stack([predict(nets[best_name], X),
                            forests[best_features].predict(X),
                            list(boostings[best_rate].staged_predict(X))[best_trees - 1]])

answers_val = answers(X_val_std)
answers_test = answers(X_test_std)
pd.DataFrame(answers_val, columns=["сеть", "лес", "бустинг"]).assign(log_S=y_val).head().round(2)

In [ ]:
# Метамодель на torch: один нейрон без функции активации, log S = w1·сеть + w2·лес + w3·бустинг + b
torch.manual_seed(0)                                                         # начальные веса воспроизводимы
meta = torch.nn.Sequential(torch.nn.Linear(3, 1))                            # 3 признака (ответы трёх моделей), 1 выход
A_t = torch.tensor(answers_val, dtype=torch.float32)                         # новые признаки
y_m = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)                  # правильные ответы

optimizer = torch.optim.LBFGS(meta.parameters(), max_iter=1000)              # обучение по всей выборке методом L-BFGS

def closure():                                                               # MSE и её производные по весам и смещению
    optimizer.zero_grad()
    mse = torch.nn.functional.mse_loss(meta(A_t), y_m)
    mse.backward()
    return mse

optimizer.step(closure)                                                      # подбор весов и смещения
w1, w2, w3 = meta[0].weight[0].tolist()                                      # веса нейрона
print(f"метамодель: log S = {w1:.2f}·сеть {w2:+.2f}·лес {w3:+.2f}·бустинг {meta[0].bias.item():+.2f}")

In [ ]:
# Стекинг на тестовой выборке и общий список
val_test["стекинг трёх моделей"] = predict(meta, answers_test)
results = pd.DataFrame({name: metrics(y_test, pred) for name, pred in val_test.items()}).T   # строка таблицы для каждой модели
results.round(3)

**Опубликованные результаты для сравнения**

| Модель | MAE | RMSE | R² |
|---|---|---|---|
| нейронные сети | 0.49 | 0.59-0.65 | 0.90-0.92 |
| случайный лес | 0.485 | 0.660 | - |
| линейные модели | 0.52-0.89 | 0.63-1.15 | 0.68-0.90 |
| консенсус нейронной сети, случайного леса и градиентного бустинга | 0.397 | 0.539 | 0.93 |

# 📌 Дополнительные материалы

**Молекулярные дескрипторы**:
* [Todeschini, Consonni, 2009](https://doi.org/10.1002/9783527628766) - *Molecular Descriptors for Chemoinformatics*, справочник по дескрипторам всех размерностей, на который опирается глава 1
* [Getting Started with the RDKit in Python](https://www.rdkit.org/docs/GettingStartedInPython.html) - работа с молекулами, поиск подструктур, расчёт дескрипторов и отпечатков

**Ансамблевые методы**:
* [Учебник по машинному обучению Яндекс](https://education.yandex.ru/handbook/ml) - главы о деревьях решений (2.4), ансамблях (2.5) и градиентном бустинге (2.6)
* scikit-learn User Guide, разделы [Decision Trees](https://scikit-learn.org/stable/modules/tree.html) и [Ensembles](https://scikit-learn.org/stable/modules/ensemble.html) - описание методов с примерами, включая стекинг и оценку сложности

**Предсказание растворимости**:
* [Sorkun, Khetan, Er, 2019](https://doi.org/10.1038/s41597-019-0151-1) - набор данных AqSolDB; [репозиторий авторов](https://github.com/mcsorkun/AqSolDB) - исходные наборы, из которых он собран, включая набор Huuskonen
* [Sorkun, Koelman, Er, 2021](https://doi.org/10.1016/j.isci.2020.101961) - модель AqSolPred, результат которой воспроизводится в главе 3
* [Huuskonen, 2000](https://doi.org/10.1021/ci9901338) - работа, в которой собрана тестовая выборка